Cell 1 — Imports

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc, json, pickle, contextlib, time, warnings, random, copy
from collections import Counter, defaultdict
from tqdm import tqdm 
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch_geometric.data import Batch
from torch_geometric.nn import GATConv, GlobalAttention, global_mean_pool
from torch.cuda.amp import GradScaler
from sklearn.metrics import (f1_score, confusion_matrix,
                             roc_auc_score, average_precision_score)
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark        = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
torch.set_float32_matmul_precision('high')

Cell 2 — Config (guideline 6c + 6d compliant)

In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SSL_METHODS_TO_RUN = ['byol', 'moco']
DATASETS_TO_RUN    = ['gotham', 'nf_bot', 'nf_unsw', 'nf_ton']

FINE_TUNE_ENCODER = True
ENCODER_LR_MULT   = 1.5
MAIN_SELECTION_METRIC = 'f1'
HIDDEN_DIM        = 128
NUM_HEADS         = 4
NUM_LAYERS        = 3
SEQUENCE_LENGTH   = 3
TFT_HIDDEN_DIM    = 128
TFT_DROPOUT       = 0.05

SEQUENCE_STRIDE   = 3
NUM_WORKERS       = 0
PREFETCH_FACTOR   = 2
BATCH_SIZE        = 64
GRAD_ACCUM_STEPS  = 256
LEARNING_RATE     = 3e-4
WEIGHT_DECAY      = 1e-3
EPOCHS            = 60
PATIENCE          = 20
FOCAL_ALPHA       = 0.25
FOCAL_GAMMA       = 2.0
POS_WEIGHT        = 1.5

MAX_NODES = 128
MAX_EDGES = 20000

ENFORCE_MAX_WINDOW_GAP   = True
DEFAULT_MAX_WINDOW_GAP_S = 60
RESUME_FROM_PARTIALS = True 
USE_TORCH_COMPILE = False
USE_PRIOR_MATCHED_THRESHOLD = False

BOOTSTRAP_N       = 1000
BOOTSTRAP_SEED    = 123
SEED              = 42

ENSEMBLE_SEEDS_DEFAULT = [1337, 2024, 7]

# ---- balanced sampler: per-dataset ----
USE_BALANCED_SAMPLER_PER_DATASET = {
    'gotham':  True,
    'nf_bot':  True,
    'nf_unsw': True,
    'nf_ton':  True,
}

# ---- few-shot small-label handling ----
SMALL_LABEL_THRESHOLD  = 300
FEWSHOT_WARMUP_SMALL   = 2
FEWSHOT_PATIENCE_SMALL = 6

WARMUP_EPOCHS  = 5
FEWSHOT_EPOCHS = 40
FEWSHOT_WARMUP = 5

FRACTIONS = [0.05, 0.25, 0.50, 1.00]

ENSEMBLE_VERBOSE = True

CKPT_EVERY_N_STEPS     = 512
KEEP_LAST_N_PARTIALS   = 5
EMPTY_CACHE_EACH_EPOCH = True

DATASET_THRESHOLDS = {
    'gotham':  0.08, 'nf_ton':  0.15, 'nf_bot':  0.10, 'nf_unsw': 0.08,
}
BLACKLIST_CLASSES = {'Unknown', 'unknown', 'UNKNOWN'}

torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
print(f"Mode: {'FINE-TUNE end-to-end' if FINE_TUNE_ENCODER else 'FROZEN ablation'}")
print(f"TFT: hidden={TFT_HIDDEN_DIM}  seq_len={SEQUENCE_LENGTH}  stride={SEQUENCE_STRIDE}")
print(f"Batch: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} accum = effective {BATCH_SIZE*GRAD_ACCUM_STEPS}")
print(f"Caps: MAX_NODES={MAX_NODES}  MAX_EDGES={MAX_EDGES}")
print(f"Selection (main): val macro-F1 merged (train-visible)")
print(f"Selection (few-shot, n<{SMALL_LABEL_THRESHOLD}): val binary ROC")
print(f"Threshold: {'prior-matched' if USE_PRIOR_MATCHED_THRESHOLD else 'F1-optimal (val)'}")
print(f"Balanced sampler per dataset: {USE_BALANCED_SAMPLER_PER_DATASET}")
print(f"Seeds: single={SEED}  ensemble={ENSEMBLE_SEEDS_DEFAULT}")
print(f"Few-shot: epochs={FEWSHOT_EPOCHS}  fractions={FRACTIONS}")
print(f"Gap enforcement: {ENFORCE_MAX_WINDOW_GAP}  (default {DEFAULT_MAX_WINDOW_GAP_S}s)")

Device: cuda
Mode: FINE-TUNE end-to-end
TFT: hidden=128  seq_len=3  stride=3
Batch: 64 x 256 accum = effective 16384
Caps: MAX_NODES=128  MAX_EDGES=20000
Selection (main): val macro-F1 merged (train-visible)
Selection (few-shot, n<300): val binary ROC
Threshold: F1-optimal (val)
Balanced sampler per dataset: {'gotham': True, 'nf_bot': True, 'nf_unsw': True, 'nf_ton': True}
Seeds: single=42  ensemble=[1337, 2024, 7]
Few-shot: epochs=40  fractions=[0.05, 0.25, 0.5, 1.0]
Gap enforcement: True  (default 60s)


Cell 3 — Dataset registry + robust autocast

In [3]:
DATASET_CONFIG = {
    'gotham':  'Gotham_processed_features_10s/graphs',
    'nf_bot':  'NFBoTIoT_v3_10s/graphs',
    'nf_unsw': 'NFUNSWNB15_v3_10s/graphs',
    'nf_ton':  'NFToNIoT_v3_10s/graphs',
}

def autocast_ctx():
    if DEVICE.type != 'cuda':
        return contextlib.nullcontext()
    try:
        return torch.amp.autocast(device_type='cuda', dtype=torch.float16)
    except Exception:
        return torch.cuda.amp.autocast(enabled=True)

print("autocast_ctx ready.")

autocast_ctx ready.


Cell 4 — Graph utilities

In [ ]:
def cap_graph(g, max_nodes=MAX_NODES, max_edges=MAX_EDGES):
    g = g.clone(); num_nodes = g.num_nodes
    if g.edge_index.size(1) > max_edges:
        keep = torch.randperm(g.edge_index.size(1))[:max_edges]
        g.edge_index = g.edge_index[:, keep]
        if g.edge_attr is not None and g.edge_attr.size(0) > 0:
            if g.edge_attr.dim() > 2:
                g.edge_attr = g.edge_attr.view(-1, g.edge_attr.size(-1))
            g.edge_attr = g.edge_attr[keep]
    if num_nodes > max_nodes:
        deg = torch.zeros(num_nodes, dtype=torch.long)
        deg.scatter_add_(0, g.edge_index[0],
                         torch.ones(g.edge_index.size(1), dtype=torch.long))
        kn = torch.topk(deg, max_nodes).indices
        mask = torch.zeros(num_nodes, dtype=torch.bool); mask[kn] = True
        s, d = g.edge_index; keep_e = mask[s] & mask[d]
        g.edge_index = g.edge_index[:, keep_e]
        if g.edge_attr is not None and g.edge_attr.size(0) > 0:
            g.edge_attr = g.edge_attr[keep_e]
        remap = torch.zeros(num_nodes, dtype=torch.long)
        remap[mask] = torch.arange(max_nodes, dtype=torch.long)
        g.edge_index = remap[g.edge_index]; g.x = g.x[mask]
    return g


def dominant_family(g, attack_threshold=0.15,
                    blacklist=('Unknown', 'unknown', 'UNKNOWN')):
    names = getattr(g, 'edge_attack_names', None)
    if not names: return 'Benign'
    c = Counter(names); total = sum(c.values())
    atk = {k: v for k, v in c.items() if k != 'Benign' and k not in blacklist}
    if not atk: return 'Benign'
    if sum(atk.values()) / max(total, 1) < attack_threshold: return 'Benign'
    return max(atk, key=atk.get)


# GAP-AWARE SEQUENCE BUILDER
def build_gap_aware_starts(graphs, seq_len, stride, max_gap_sec,
                           enforce=True, verbose=True):
    n = len(graphs)
    if n < seq_len:
        return np.array([], dtype=np.int64)

    stride_only = np.arange(0, n - seq_len + 1, stride, dtype=np.int64)

    if not enforce:
        return stride_only

    has_time = hasattr(graphs[0], 'window_start_unix')
    if not has_time:
        if verbose:
            print("  [gap] graphs lack .window_start_unix — gap check skipped")
        return stride_only

    times = np.array([float(g.window_start_unix) for g in graphs], dtype=np.float64)
    # Adjacent gaps (length n-1)
    gaps = np.diff(times)
    ok_adj = (gaps <= float(max_gap_sec)).astype(np.int32)   # 1 if gap OK

    # For start s, we require ok_adj[s : s+seq_len-1] to be all 1 (seq_len-1 entries)
    cum = np.concatenate([[0], np.cumsum(ok_adj)])
    window_sum = cum[stride_only + seq_len - 1] - cum[stride_only]
    keep = window_sum == (seq_len - 1)
    kept = stride_only[keep]

    if verbose:
        n_rejected = int((~keep).sum())
        print(f"  [gap] stride candidates {len(stride_only)}  "
              f"rejected-by-gap {n_rejected}  kept {len(kept)}  "
              f"(max_gap={max_gap_sec}s)")

    if len(kept) < 10 and len(stride_only) >= 10:
        if verbose:
            print(f"  [gap][warn] only {len(kept)} sequences survive — "
                  f"relaxing gap constraint to stride-only")
        return stride_only

    return kept

Cell 5 — Model classes (6c: VSN + Interpretable MHA + GRN + LSTM)

In [5]:
class GATEncoder(nn.Module):
    def __init__(self, in_ch, hid, heads, layers, edge_dim):
        super().__init__()
        self.convs = nn.ModuleList(); self.bns = nn.ModuleList()
        self.convs.append(GATConv(in_ch, hid, heads=heads, edge_dim=edge_dim,
                                  concat=True, dropout=0.0))
        self.bns.append(nn.BatchNorm1d(hid * heads))
        for _ in range(layers - 2):
            self.convs.append(GATConv(hid * heads, hid, heads=heads,
                                      edge_dim=edge_dim, concat=True, dropout=0.0))
            self.bns.append(nn.BatchNorm1d(hid * heads))
        self.convs.append(GATConv(hid * heads, hid, heads=1, edge_dim=edge_dim,
                                  concat=False, dropout=0.0))
        self.bns.append(nn.BatchNorm1d(hid))
        self.edge_dim = edge_dim
        self.pool_gate = nn.Sequential(
            nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, 1))

    def forward(self, x, ei, ea=None, b=None, return_nodes=False):
        for i, c in enumerate(self.convs):
            x = c(x, ei, edge_attr=ea) if (ea is not None and self.edge_dim > 0) \
                else c(x, ei)
            x = F.relu(self.bns[i](x))
        h = GlobalAttention(self.pool_gate)(x, b)   # SAME on both paths
        if return_nodes:
            return x, h
        return h


class GatedResidualNetwork(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.1):
        super().__init__()
        self.fc1  = nn.Linear(input_dim, hidden_dim); self.elu = nn.ELU()
        self.fc2  = nn.Linear(hidden_dim, output_dim); self.drop = nn.Dropout(dropout)
        self.gate = nn.Linear(hidden_dim, output_dim); self.norm = nn.LayerNorm(output_dim)
        self.skip = nn.Linear(input_dim, output_dim) if input_dim != output_dim else None

    def forward(self, x):
        z = self.elu(self.fc1(x))
        h = self.drop(self.fc2(z))
        g = torch.sigmoid(self.gate(z))
        h = g * h
        if self.skip is not None:
            x = self.skip(x)
        return self.norm(x + h)


class VariableSelectionNetwork(nn.Module):
    def __init__(self, var_dims, hidden_dim, dropout=0.1):
        super().__init__()
        self.num_vars = len(var_dims)
        self.var_grns = nn.ModuleList(
            [GatedResidualNetwork(d, hidden_dim, hidden_dim, dropout) for d in var_dims])
        self.weight_grn = GatedResidualNetwork(
            sum(var_dims), hidden_dim, self.num_vars, dropout)

    def forward(self, vars_list):
        processed = [grn(v) for grn, v in zip(self.var_grns, vars_list)]
        flat = torch.cat(vars_list, dim=-1)
        w = F.softmax(self.weight_grn(flat), dim=-1)
        stacked = torch.stack(processed, dim=2)
        return (stacked * w.unsqueeze(-1)).sum(dim=2), w


class InterpretableMHA(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout=0.1):
        super().__init__()
        self.h = hidden_dim; self.nh = num_heads; self.hd = hidden_dim // num_heads
        assert self.hd * num_heads == hidden_dim
        self.q = nn.Linear(hidden_dim, hidden_dim); self.k = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, hidden_dim); self.o = nn.Linear(hidden_dim, hidden_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        s = torch.matmul(q, k.transpose(-2, -1)) / (self.hd ** 0.5)
        a = self.drop(F.softmax(s, dim=-1))
        c = torch.matmul(a, v).transpose(1, 2).contiguous().view(B, T, self.h)
        return self.o(c), a.mean(dim=1)


class TemporalFusionModuleV2(nn.Module):
    def __init__(self, graph_emb_dim, struct_dim, hidden_dim, num_heads, dropout=0.1):
        super().__init__()
        self.proj_graph  = nn.Linear(graph_emb_dim, hidden_dim)
        self.proj_struct = nn.Linear(struct_dim, hidden_dim)
        self.vsn  = VariableSelectionNetwork([hidden_dim, hidden_dim], hidden_dim, dropout)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers=2,
                            batch_first=True, dropout=dropout)
        self.attn = InterpretableMHA(hidden_dim, num_heads, dropout)
        self.grn1 = GatedResidualNetwork(hidden_dim, hidden_dim, hidden_dim, dropout)
        self.grn2 = GatedResidualNetwork(hidden_dim, hidden_dim, hidden_dim, dropout)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, graph_emb, struct_feats):
        v_g = self.proj_graph(graph_emb); v_s = self.proj_struct(struct_feats)
        selected, vsn_w = self.vsn([v_g, v_s])
        lstm_out, _ = self.lstm(selected)
        attn_out, attn_w = self.attn(lstm_out)
        out = self.grn1(attn_out); out = self.grn2(out)
        out = self.norm(selected + out)
        return out[:, -1, :], attn_w, vsn_w


class MC_STGAT_IDS(nn.Module):
    def __init__(self, graph_emb_dim, struct_dim, tft_hidden, num_classes,
                 num_heads=4, dropout=0.1):
        super().__init__()
        self.temporal = TemporalFusionModuleV2(graph_emb_dim, struct_dim,
                                               tft_hidden, num_heads, dropout)
        self.binary_head = nn.Sequential(
            nn.Linear(tft_hidden, tft_hidden // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(tft_hidden // 2, 1))
        self.multi_head = nn.Sequential(
            nn.Linear(tft_hidden, tft_hidden // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(tft_hidden // 2, num_classes))

    def forward(self, graph_emb, struct_feats):
        temporal_emb, attn_w, vsn_w = self.temporal(graph_emb, struct_feats)
        return (self.binary_head(temporal_emb).squeeze(-1),
                self.multi_head(temporal_emb), attn_w, vsn_w)


class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, pos_weight=1.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
        self.register_buffer('pos_weight', torch.tensor(float(pos_weight)))
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none',
                                                 pos_weight=self.pos_weight)
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        return (self.alpha * (1 - p_t) ** self.gamma * bce).mean()


class CombinedLoss(nn.Module):
    def __init__(self, w_multi=0.7, pos_weight=1.5, focal_alpha=0.25, focal_gamma=2.0):
        super().__init__()
        self.w_multi = w_multi
        self.binary_loss = BinaryFocalLoss(focal_alpha, focal_gamma, pos_weight)
        self.focal_alpha = focal_alpha; self.focal_gamma = focal_gamma
    def forward(self, bin_logits, multi_logits, y_bin, y_multi):
        l_bin = self.binary_loss(bin_logits, y_bin)
        logp = F.log_softmax(multi_logits, dim=-1)
        logp_t = logp.gather(1, y_multi.unsqueeze(1)).squeeze(1)
        p_t = logp_t.exp()
        l_multi = (-self.focal_alpha * (1 - p_t) ** self.focal_gamma * logp_t).mean()
        return l_bin + self.w_multi * l_multi, l_bin.item(), l_multi.item()

Cell 6 — Sequence dataset + collate.

**Key change vs. frozen version:** returns *PyG graphs per sequence*, not cached embeddings.

In [6]:
class GraphSequenceDataset(Dataset):
    def __init__(self, graphs, starts, seq_len, y_bin_seq, y_multi_seq, struct_feats):
        self.graphs      = graphs
        self.starts      = np.asarray(starts)
        self.seq_len     = seq_len
        self.y_bin_seq   = y_bin_seq
        self.y_multi_seq = y_multi_seq
        self.struct      = struct_feats
    def __len__(self): return len(self.starts)
    def __getitem__(self, i):
        s = int(self.starts[i]); e = s + self.seq_len
        return (self.graphs[s:e], self.struct[s:e],
                self.y_bin_seq[i], self.y_multi_seq[i])


def collate_graph_sequences(batch):

    seq_graphs_list, seq_struct_list, y_bin_list, y_multi_list = zip(*batch)
    B = len(seq_graphs_list)
    T = len(seq_graphs_list[0])

    flat_graphs = []
    for seq in seq_graphs_list:
        flat_graphs.extend(seq)
    flat_batch = Batch.from_data_list(flat_graphs)

    struct_tensor = torch.stack(
        [torch.as_tensor(s, dtype=torch.float32) for s in seq_struct_list], dim=0)

    return (flat_batch, B, T, struct_tensor,
            torch.tensor(y_bin_list,   dtype=torch.float32),
            torch.tensor(y_multi_list, dtype=torch.long))

Cell 7 — Structural feature precompute (cheap, no grad)

In [7]:
def structural_features(g):
    x  = g.x.numpy()
    ea = g.edge_attr.numpy() if g.edge_attr is not None else np.zeros((0, 0))
    ei = g.edge_index.numpy()
    N = x.shape[0]; E = ei.shape[1]
    deg = np.bincount(ei[0], minlength=N) if E > 0 else np.zeros(N)
    base = np.array([float(N), float(E),
                     float(deg.mean()) if N > 0 else 0.0,
                     float(np.log1p(E))], dtype=np.float32)
    nm = x.mean(0) if N > 0 else np.zeros(x.shape[1], dtype=np.float32)
    ns = x.std(0)  if N > 0 else np.zeros(x.shape[1], dtype=np.float32)
    em = ea.mean(0) if E > 0 and ea.ndim == 2 else np.zeros(ea.shape[-1] if ea.ndim == 2 else 0,
                                                              dtype=np.float32)
    return np.concatenate([base, nm, ns, em]).astype(np.float32)


def precompute_struct(graphs):
    return torch.tensor(np.stack([structural_features(g) for g in graphs]),
                        dtype=torch.float32)

In [ ]:
# Logging / checkpoint / threshold helpers
def _fmt_time(seconds):
    if seconds < 60:
        return f"{seconds:5.1f}s"
    m, s = divmod(seconds, 60)
    return f"{int(m):02d}m{s:04.1f}s"

def _bar(value, width=20, vmin=0.5, vmax=1.0):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return " " * width
    frac = float(np.clip((value - vmin) / max(vmax - vmin, 1e-9), 0.0, 1.0))
    n = int(round(frac * width))
    return " " * n + "·" * (width - n)

def _print_epoch_header(tag, total_epochs):
    print(f"  {'ep':>4}  {'train':>8}  {'bin':>7}  {'multi':>7}  "
          f"{'gnorm':>7}  {'lr':>9}  |  "
          f"{'valROC':>7} {'valPR':>7} {'valF1':>7} {'valmF1':>8}  "
          f"{'ROC bar':<20}  {'time':>7}  {'pat':>3}  note")
    print("  " + "-" * 130)

def _print_epoch_row(ep, total_epochs, tr, val, gnorm, lr,
                     patience_left, best_macro_f1, elapsed, note=""):
    print(f"  {ep:>4}  "
          f"{tr['total']:>8.4f}  {tr['binary']:>7.4f}  {tr['multi']:>7.4f}  "
          f"{gnorm:>7.3f}  {lr:>9.2e}  |  "
          f"{val['roc']:>7.4f} {val['pr']:>7.4f} {val['f1']:>7.4f} "
          f"{val['macro_f1_merged']:>8.4f}  "
          f"{_bar(val['roc']):<20}  "
          f"{_fmt_time(elapsed):>7}  {patience_left:>3}  {note}")

def _optimal_threshold(prob, y):
    """F1-optimal threshold on the reference set. Kept for comparison."""
    thrs = np.linspace(0.05, 0.95, 91)
    best_t, best_f1 = 0.5, -1.0
    for t in thrs:
        f1 = f1_score(y, (prob >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)

def _prior_matched_threshold(prob, y):
    target_rate = float(np.asarray(y).mean())
    if target_rate <= 0.0 or target_rate >= 1.0:
        return 0.5
    return float(np.quantile(prob, 1.0 - target_rate))

def _binary_metrics_at(prob, y, thr):
    pred = (prob >= thr).astype(int)
    f1  = float(f1_score(y, pred, average='binary', zero_division=0))
    cm  = confusion_matrix(y, pred, labels=[0, 1])
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    return {
        'f1':  f1,
        'fpr': float(cmn[0, 1]),
        'fnr': float(cmn[1, 0]),
        'n_pred_attack': int(pred.sum()),
        'pred_attack_rate': float(pred.mean()),
    }

def _per_class_score_analysis(pb_test, test_labels_seq, class_names, thr):
    rows = []
    for cls in class_names:
        if cls == 'Benign':
            continue
        mask = (test_labels_seq == cls) | (test_labels_seq == 'Benign')
        y_sel = (test_labels_seq[mask] != 'Benign').astype(int)
        p_sel = pb_test[mask]
        if len(set(y_sel)) < 2:
            continue
        try:
            roc = float(roc_auc_score(y_sel, p_sel))
            pr  = float(average_precision_score(y_sel, p_sel))
        except ValueError:
            continue
        pos = (y_sel == 1)
        rows.append({
            'class': cls, 'n_test': int(pos.sum()),
            'roc_vs_benign': roc, 'pr_vs_benign': pr,
            'mean_prob_attack': float(p_sel[pos].mean()) if pos.sum() > 0 else 0.0,
            'recall_at_thr': float((p_sel[pos] >= thr).sum() / max(int(pos.sum()), 1)),
            'missed_at_thr': int((p_sel[pos] < thr).sum()),
        })
    rows.sort(key=lambda r: -r['roc_vs_benign'])
    return rows

def _print_per_class_table(rows, thr, title):
    if not rows:
        print(f"  {title}: no rows")
        return
    print(f"\n  {title} — thr={thr:.2f}")
    print(f"  {'attack class':<28} {'n':>6} {'ROC':>7} {'PR':>7} "
          f"{'meanP':>7} {'recall':>7} {'missed':>7}")
    for r in rows:
        print(f"  {r['class'][:28]:<28} {r['n_test']:>6} "
              f"{r['roc_vs_benign']:>7.4f} {r['pr_vs_benign']:>7.4f} "
              f"{r['mean_prob_attack']:>7.3f} {r['recall_at_thr']:>7.3f} "
              f"{r['missed_at_thr']:>7}")

def _prune_partials(ckpt_dir, tag, keep=5):
    try:
        files = sorted(
            f for f in os.listdir(ckpt_dir)
            if f.startswith(tag + "_ep") and f.endswith(".pth") and "_step" in f
        )
    except FileNotFoundError:
        return
    to_del = files if keep <= 0 else files[:-keep]
    for f in to_del:
        try:
            os.remove(os.path.join(ckpt_dir, f))
        except OSError:
            pass

def _save_partial_checkpoint(ckpt_dir, tag, epoch, step, enc, tft, opt, scaler,
                             running_loss=None, extra=None):
    os.makedirs(ckpt_dir, exist_ok=True)
    path = os.path.join(ckpt_dir, f"{tag}_ep{int(epoch):02d}_step{int(step):05d}.pth")
    payload = {
        'epoch': int(epoch), 'step': int(step), 'phase': 'mid_epoch',
        'fine_tuned_encoder': bool(FINE_TUNE_ENCODER),
        'encoder_state_dict': enc.state_dict(),
        'tft_state_dict': tft.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
        'running_loss': float(running_loss) if running_loss is not None else None,
    }
    try:
        payload['scaler_state_dict'] = scaler.state_dict()
    except Exception:
        pass
    if extra:
        payload.update(extra)
    torch.save(payload, path)
    _prune_partials(ckpt_dir, tag, keep=KEEP_LAST_N_PARTIALS)
    return path

def _save_phase_checkpoint(ckpt_dir, tag, epoch, phase, enc, tft, opt,
                           sched=None, scaler=None, metrics=None):
    os.makedirs(ckpt_dir, exist_ok=True)
    path = os.path.join(ckpt_dir, f"{tag}_ep{int(epoch):02d}_{phase}.pth")
    payload = {
        'epoch': int(epoch), 'phase': phase,
        'fine_tuned_encoder': bool(FINE_TUNE_ENCODER),
        'encoder_state_dict': enc.state_dict(),
        'tft_state_dict': tft.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
    }
    if sched is not None:
        try: payload['scheduler_state_dict'] = sched.state_dict()
        except Exception: pass
    if scaler is not None:
        try: payload['scaler_state_dict'] = scaler.state_dict()
        except Exception: pass
    if metrics is not None:
        payload['metrics'] = metrics
    torch.save(payload, path)
    return path

def _flush_report(out_dir, filename, payload):
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, filename)
    tmp  = path + ".tmp"
    with open(tmp, 'w') as f:
        json.dump(payload, f, indent=2, default=str)
    os.replace(tmp, path)
    print(f"    [flush] {path}")
    return path

def _encode_sequence(encoder, flat_batch, B, T, device, frozen):
    ctx = torch.no_grad() if frozen else contextlib.nullcontext()
    with ctx:
        flat_batch = flat_batch.to(device, non_blocking=True)
        out = encoder(flat_batch.x, flat_batch.edge_index,
                      flat_batch.edge_attr, flat_batch.batch)
        h = out[-1] if isinstance(out, tuple) else out
        return h.view(B, T, -1)

def _val_metrics(pb_val, pm_val, yb_val, ym_val,
                 thr=0.5,
                 train_visible_idx=None,
                 class_names=None,
                 dataset=None):
    y = yb_val.astype(int)
    try:
        roc = float(roc_auc_score(y, pb_val))
    except ValueError:
        roc = 0.5
    try:
        pr = float(average_precision_score(y, pb_val))
    except ValueError:
        pr = 0.0
    f1_bin = float(f1_score(y, (pb_val >= thr).astype(int),
                            average='binary', zero_division=0))

    ym_pred = pm_val.argmax(-1)

    macro_f1_all = float(f1_score(ym_val, ym_pred,
                                  average='macro', zero_division=0))
    weighted_f1  = float(f1_score(ym_val, ym_pred,
                                  average='weighted', zero_division=0))

    if train_visible_idx is None or len(train_visible_idx) < 2:
        macro_f1_visible = macro_f1_all
    else:
        mask = np.isin(ym_val, train_visible_idx)
        if mask.sum() < 2 or len(np.unique(ym_val[mask])) < 2:
            macro_f1_visible = macro_f1_all
        else:
            macro_f1_visible = float(f1_score(
                ym_val[mask], ym_pred[mask],
                average='macro', zero_division=0))

    if class_names is None or dataset is None:
        macro_f1_merged = macro_f1_visible
    else:
        ym_val_m  = np.array([map_to_family(class_names[i], dataset) for i in ym_val])
        ym_pred_m = np.array([map_to_family(class_names[i], dataset) for i in ym_pred])
        train_fams = sorted({map_to_family(class_names[i], dataset)
                             for i in (train_visible_idx
                                       if train_visible_idx is not None
                                       else np.unique(ym_val))})
        mask_m = np.isin(ym_val_m, list(train_fams))
        if mask_m.sum() < 2 or len(np.unique(ym_val_m[mask_m])) < 2:
            macro_f1_merged = float(f1_score(ym_val_m, ym_pred_m,
                                             average='macro', zero_division=0))
        else:
            macro_f1_merged = float(f1_score(ym_val_m[mask_m], ym_pred_m[mask_m],
                                             average='macro', zero_division=0))

    return {
        'roc':              roc,
        'pr':               pr,
        'f1':               f1_bin,
        'macro_f1':         macro_f1_all,
        'macro_f1_visible': macro_f1_visible,
        'macro_f1_merged':  macro_f1_merged,
        'weighted_f1':      weighted_f1,
    }

print("TFT helpers ready.")

TFT helpers ready.


Cell 8 — run_dataset: end-to-end fine-tune + attention extraction

In [ ]:
def run_dataset(DATASET_NAME, SSL_METHOD, ensemble_seeds=None, force_retrain=False,
                resume_from_partials=None):
    if resume_from_partials is None:
        resume_from_partials = globals().get('RESUME_FROM_PARTIALS', True)

    if ensemble_seeds is None:
        ensemble_seeds = list(ENSEMBLE_SEEDS_DEFAULT)
    else:
        ensemble_seeds = list(ensemble_seeds)
    if SEED in ensemble_seeds:
        ensemble_seeds = [s for s in ensemble_seeds if s != SEED]

    use_balanced_sampler = USE_BALANCED_SAMPLER_PER_DATASET.get(DATASET_NAME, True)

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    SAVE_PARTIALS = CKPT_EVERY_N_STEPS > 0

    print(f"Single seed: {SEED}   Ensemble seeds: {ensemble_seeds} "
          f"({len(ensemble_seeds)} members)")
    print(f"Balanced sampler: {use_balanced_sampler}")
    print(f"Resume from partials: {resume_from_partials}  force_retrain: {force_retrain}")

    DATA_DIR       = DATASET_CONFIG[DATASET_NAME]
    CHECKPOINT_DIR = os.path.join(DATA_DIR, f"ssl_checkpoints_{SSL_METHOD}")
    mode_tag       = "finetune" if FINE_TUNE_ENCODER else "frozen"
    PHASE3_DIR     = os.path.join(DATA_DIR, f"phase3_checkpoints_{SSL_METHOD}_{mode_tag}")
    ATTN_DIR       = os.path.join(PHASE3_DIR, "attention")
    PARTIAL_DIR    = os.path.join(PHASE3_DIR, "partial_checkpoints")
    os.makedirs(PHASE3_DIR, exist_ok=True)
    os.makedirs(ATTN_DIR, exist_ok=True)
    os.makedirs(PARTIAL_DIR, exist_ok=True)

    CKPT_FILE   = os.path.join(PHASE3_DIR, "mc_stgat_ids.pth")
    REPORT_FILE = os.path.join(PHASE3_DIR, "phase3_report.json")
    ENC_FILE    = os.path.join(CHECKPOINT_DIR, "ssl_encoder_best.pth")

    if not os.path.isfile(ENC_FILE):
        print(f"\n[SKIP] {DATASET_NAME} [{SSL_METHOD.upper()}]: no SSL encoder at {ENC_FILE}")
        return {'_skipped_no_ssl': True, 'dataset': DATASET_NAME, 'ssl_method': SSL_METHOD}

    # Full cache: both final ckpt and report already exist -> nothing to do.
    if (not force_retrain) and os.path.isfile(CKPT_FILE) and os.path.isfile(REPORT_FILE):
        print(f"[SKIP] {DATASET_NAME} [{SSL_METHOD.upper()}/{mode_tag}]: cached")
        with open(REPORT_FILE) as f: r = json.load(f)
        r['_loaded_from_cache'] = True
        return r

    # ---- resume-stage markers ----
    BEST_MAIN = os.path.join(PARTIAL_DIR, f"best_{DATASET_NAME}_{SSL_METHOD}.pth")
    MAIN_JSON = os.path.join(PHASE3_DIR, "phase3_main_seed_report.json")

    def _fewshot_json(frac):  return os.path.join(PHASE3_DIR, f"phase3_fewshot_{int(frac*100):03d}.json")
    def _ens_seed_json(seed): return os.path.join(PHASE3_DIR, f"phase3_ensemble_seed{seed}.json")
    def _ens_seed_ckpt(seed): return os.path.join(PARTIAL_DIR, f"best_{DATASET_NAME}_{SSL_METHOD}_seed{seed}.pth")

    main_done = (resume_from_partials and not force_retrain
                 and os.path.isfile(BEST_MAIN)
                 and os.path.isfile(MAIN_JSON))
    _fs_markers = {float(f): os.path.isfile(_fewshot_json(f)) for f in FRACTIONS}
    _ens_markers = {int(s): (os.path.isfile(_ens_seed_json(s))
                              and os.path.isfile(_ens_seed_ckpt(s)))
                    for s in ensemble_seeds}
    print(f"[resume] main seed done: {main_done}")
    print(f"[resume] few-shot markers: {_fs_markers}")
    print(f"[resume] ensemble markers: {_ens_markers}")

    print(f"\n{'='*70}")
    print(f"Phase 3 · {DATASET_NAME} · SSL={SSL_METHOD.upper()} · mode={mode_tag.upper()}")
    print(f"  encoder {'FINE-TUNED end-to-end' if FINE_TUNE_ENCODER else 'FROZEN'}")
    print(f"  window stride={SEQUENCE_STRIDE}  workers={NUM_WORKERS}")
    print(f"  selection={MAIN_SELECTION_METRIC} | "
          f"threshold={'prior-matched' if USE_PRIOR_MATCHED_THRESHOLD else 'F1-optimal'}")
    print(f"  balanced sampler={use_balanced_sampler}")
    print(f"  gap enforcement={ENFORCE_MAX_WINDOW_GAP}")
    print(f"  partial ckpts={'ON' if SAVE_PARTIALS else 'OFF'}")
    print(f"{'='*70}")

    torch.manual_seed(SEED); np.random.seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    attack_threshold = DATASET_THRESHOLDS[DATASET_NAME]

    def load_graphs(s):
        with open(os.path.join(DATA_DIR, f"{s}_graphs.pkl"), 'rb') as f:
            return pickle.load(f)

    print("Loading + capping graphs...")
    train_graphs = [cap_graph(g) for g in load_graphs('train')]
    val_graphs   = [cap_graph(g) for g in load_graphs('val')]
    test_graphs  = [cap_graph(g) for g in load_graphs('test')]
    print(f"  loaded: train={len(train_graphs)} val={len(val_graphs)} test={len(test_graphs)}")

    with open(os.path.join(DATA_DIR, "norm_stats.json")) as f:
        ns = json.load(f)
    N_CONT    = ns["n_continuous_edge_feats"]
    node_mean = torch.tensor(ns["node_mean"], dtype=torch.float)
    node_std  = torch.tensor(ns["node_std"],  dtype=torch.float)
    edge_mean = torch.tensor(ns["edge_mean"], dtype=torch.float)
    edge_std  = torch.tensor(ns["edge_std"],  dtype=torch.float)

    MAX_WINDOW_GAP_SEC = float(ns.get('max_window_gap_sec', DEFAULT_MAX_WINDOW_GAP_S))
    print(f"  max_window_gap_sec (from norm_stats): {MAX_WINDOW_GAP_SEC}")

    def normalise(graphs):
        for g in graphs:
            g.x = torch.clamp((g.x - node_mean) / (node_std + 1e-8), -10, 10)
            if g.edge_attr is not None and g.edge_attr.size(0) > 0:
                if g.edge_attr.dim() > 2:
                    g.edge_attr = g.edge_attr.view(-1, g.edge_attr.size(-1))
                c, p = g.edge_attr[:, :N_CONT], g.edge_attr[:, N_CONT:]
                c = torch.clamp((c - edge_mean) / (edge_std + 1e-8), -10, 10)
                g.edge_attr = torch.cat([c, p], dim=1)
            if g.edge_attr is None or g.edge_attr.size(0) == 0:
                g.edge_attr = torch.zeros((g.edge_index.size(1), N_CONT), dtype=torch.float)
            g.x = torch.nan_to_num(g.x, nan=0.0, posinf=0.0, neginf=0.0)
            g.edge_attr = torch.nan_to_num(g.edge_attr, nan=0.0, posinf=0.0, neginf=0.0)
    normalise(train_graphs); normalise(val_graphs); normalise(test_graphs)

    train_labels = np.array([dominant_family(g, attack_threshold) for g in train_graphs])
    val_labels   = np.array([dominant_family(g, attack_threshold) for g in val_graphs])
    test_labels  = np.array([dominant_family(g, attack_threshold) for g in test_graphs])

    class_names  = sorted(set(train_labels.tolist()) |
                          set(val_labels.tolist()) |
                          set(test_labels.tolist()))
    label_to_idx = {c: i for i, c in enumerate(class_names)}
    NUM_CLASSES  = len(class_names)

    to_bin = lambda a: (a != 'Benign').astype(np.float32)
    train_bin = to_bin(train_labels); val_bin = to_bin(val_labels); test_bin = to_bin(test_labels)
    train_multi = np.array([label_to_idx[c] for c in train_labels], dtype=np.int64)
    val_multi   = np.array([label_to_idx[c] for c in val_labels],   dtype=np.int64)
    test_multi  = np.array([label_to_idx[c] for c in test_labels],  dtype=np.int64)

    train_visible_idx = np.unique(train_multi)
    train_visible_fams = sorted({map_to_family(class_names[i], DATASET_NAME)
                                 for i in train_visible_idx})
    print(f"\nTrain-visible fine classes : {len(train_visible_idx)} / {NUM_CLASSES}")
    print(f"Train-visible families     : {len(train_visible_fams)}  {train_visible_fams}")

    print(f"\nClasses ({NUM_CLASSES}): {class_names}")
    print(f"Train: benign={int((train_bin==0).sum())} attack={int((train_bin==1).sum())}")
    print(f"Val  : benign={int((val_bin==0).sum())}   attack={int((val_bin==1).sum())}")
    print(f"Test : benign={int((test_bin==0).sum())}  attack={int((test_bin==1).sum())}")

    ckpt = torch.load(ENC_FILE, map_location=DEVICE)
    ssl_objective = ckpt.get('ssl_objective', SSL_METHOD)
    print(f"Loaded SSL encoder [{SSL_METHOD.upper()}]: "
          f"epoch {ckpt.get('epoch','?')}  "
          f"val_roc {ckpt.get('val_roc_auc', float('nan')):.4f}")

    def fresh_encoder():
        enc = GATEncoder(ckpt['node_dim'], ckpt['hidden_dim'], ckpt['num_heads'],
                         ckpt['num_layers'], ckpt['edge_dim']).to(DEVICE)
        enc.load_state_dict(ckpt['encoder_state_dict'])
        if not FINE_TUNE_ENCODER:
            enc.eval()
            for p in enc.parameters(): p.requires_grad = False
        if USE_TORCH_COMPILE:
            try:
                enc = torch.compile(enc, mode="default")
                print("  encoder: torch.compile OK")
            except Exception as e:
                print(f"  encoder: torch.compile failed ({e}); using eager")
        return enc

    str_train = precompute_struct(train_graphs)
    str_val   = precompute_struct(val_graphs)
    str_test  = precompute_struct(test_graphs)
    NUM_STRUCT_FEATS = str_train.size(1)
    print(f"struct_dim = {NUM_STRUCT_FEATS}")

    print("\nBuilding gap-aware sequence starts...")
    train_starts = build_gap_aware_starts(train_graphs, SEQUENCE_LENGTH,
                                          SEQUENCE_STRIDE, MAX_WINDOW_GAP_SEC,
                                          enforce=ENFORCE_MAX_WINDOW_GAP)
    val_starts   = build_gap_aware_starts(val_graphs,   SEQUENCE_LENGTH,
                                          SEQUENCE_STRIDE, MAX_WINDOW_GAP_SEC,
                                          enforce=ENFORCE_MAX_WINDOW_GAP)
    test_starts  = build_gap_aware_starts(test_graphs,  SEQUENCE_LENGTH,
                                          SEQUENCE_STRIDE, MAX_WINDOW_GAP_SEC,
                                          enforce=ENFORCE_MAX_WINDOW_GAP)
    print(f"Sequences: train={len(train_starts)} val={len(val_starts)} test={len(test_starts)}")

    def build_seq_labels(y, seq_len, starts):
        return np.array([y[s + seq_len - 1] for s in starts])

    train_y_bin   = build_seq_labels(train_bin, SEQUENCE_LENGTH, train_starts)
    val_y_bin     = build_seq_labels(val_bin,   SEQUENCE_LENGTH, val_starts)
    test_y_bin    = build_seq_labels(test_bin,  SEQUENCE_LENGTH, test_starts)
    train_y_multi = build_seq_labels(train_multi, SEQUENCE_LENGTH, train_starts)
    val_y_multi   = build_seq_labels(val_multi,   SEQUENCE_LENGTH, val_starts)
    test_y_multi  = build_seq_labels(test_multi,  SEQUENCE_LENGTH, test_starts)
    test_labels_seq = build_seq_labels(test_labels, SEQUENCE_LENGTH, test_starts)

    def make_loader(graphs, starts, y_bin, y_multi, struct, shuffle, sampler=None):
        ds = GraphSequenceDataset(graphs, starts, SEQUENCE_LENGTH, y_bin, y_multi, struct)
        kw = dict(
            batch_size=BATCH_SIZE,
            collate_fn=collate_graph_sequences,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE.type == 'cuda'),
        )
        if NUM_WORKERS > 0:
            kw['persistent_workers'] = True
            kw['prefetch_factor']    = PREFETCH_FACTOR
        if sampler is not None:
            return DataLoader(ds, sampler=sampler, **kw)
        return DataLoader(ds, shuffle=shuffle, **kw)

    if use_balanced_sampler:
        n_ben = int((train_y_bin == 0).sum()); n_atk = int((train_y_bin == 1).sum())
        w = np.where(train_y_bin == 0, 1.0/max(n_ben,1), 1.0/max(n_atk,1)).astype(np.float64)
        w = w / w.sum() * len(train_y_bin)
        sampler = WeightedRandomSampler(weights=torch.tensor(w, dtype=torch.double),
                                        num_samples=len(train_y_bin), replacement=True)
        train_loader = make_loader(train_graphs, train_starts, train_y_bin, train_y_multi,
                                   str_train, False, sampler=sampler)
    else:
        train_loader = make_loader(train_graphs, train_starts, train_y_bin, train_y_multi,
                                   str_train, True)
    val_loader  = make_loader(val_graphs,  val_starts,  val_y_bin,  val_y_multi,  str_val,  False)
    test_loader = make_loader(test_graphs, test_starts, test_y_bin, test_y_multi, str_test, False)

    if not FINE_TUNE_ENCODER:
        print("\nPre-caching frozen encoder embeddings (one-time cost)...")
        t_cache = time.time()
        _enc0 = fresh_encoder()

        def _materialize(loader, name):
            items = []
            for i, (flat_batch, B, T, struct_seq, y_bin, y_multi) in enumerate(loader):
                with torch.inference_mode():
                    emb = _encode_sequence(_enc0, flat_batch, B, T, DEVICE, frozen=True)
                items.append((emb.half().cpu(),
                              struct_seq.cpu(),
                              y_bin.cpu(),
                              y_multi.cpu()))
                if (i + 1) % 50 == 0:
                    print(f"    [{name}] {i+1}/{len(loader)} batches cached")
            return items

        cache_train = _materialize(train_loader, 'train')
        cache_val   = _materialize(val_loader,   'val')
        cache_test  = _materialize(test_loader,  'test')
        del _enc0
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
        print(f"  embedding cache built in {_fmt_time(time.time() - t_cache)} "
              f"(train={len(cache_train)} val={len(cache_val)} test={len(cache_test)} batches, fp16)")
    else:
        cache_train = cache_val = cache_test = None
        print("\nFINE-TUNE mode: embedding cache disabled (encoder changes each epoch)")

    def _train_data(shuffle=False, subset_idx=None):
        if cache_train is not None:
            items = cache_train if subset_idx is None else [cache_train[i] for i in subset_idx]
            if shuffle:
                order = np.random.permutation(len(items))
                items = [items[i] for i in order]
            return items
        return train_loader

    def _val_data():  return cache_val  if cache_val  is not None else val_loader
    def _test_data(): return cache_test if cache_test is not None else test_loader

    criterion = CombinedLoss(w_multi=0.7, pos_weight=POS_WEIGHT,
                             focal_alpha=FOCAL_ALPHA, focal_gamma=FOCAL_GAMMA).to(DEVICE)

    def build_tft():
        return MC_STGAT_IDS(HIDDEN_DIM, NUM_STRUCT_FEATS, TFT_HIDDEN_DIM,
                            NUM_CLASSES, NUM_HEADS, TFT_DROPOUT).to(DEVICE)

    def make_optimizer(enc, tft):
        groups = []
        if FINE_TUNE_ENCODER:
            groups.append({'params': enc.parameters(), 'lr': LEARNING_RATE * ENCODER_LR_MULT})
        groups.append({'params': tft.parameters(), 'lr': LEARNING_RATE})
        return torch.optim.Adam(groups, weight_decay=WEIGHT_DECAY)

    def _save_partial_light(ckpt_dir, tag, epoch, step, enc, tft):
        if not SAVE_PARTIALS:
            return
        payload = {'tag': tag, 'epoch': epoch, 'step': step,
                   'tft_state_dict': {k: v.detach().cpu() for k, v in tft.state_dict().items()}}
        if FINE_TUNE_ENCODER:
            payload['encoder_state_dict'] = {k: v.detach().cpu()
                                             for k, v in enc.state_dict().items()}
        torch.save(payload, os.path.join(ckpt_dir, f"partial_{tag}_e{epoch}_s{step}.pth"))

    def _save_best_light(ckpt_dir, tag, epoch, enc, tft, metrics):
        payload = {'tag': tag, 'epoch': epoch, 'metrics': metrics,
                   'tft_state_dict': {k: v.detach().cpu() for k, v in tft.state_dict().items()}}
        if FINE_TUNE_ENCODER:
            payload['encoder_state_dict'] = {k: v.detach().cpu()
                                             for k, v in enc.state_dict().items()}
        torch.save(payload, os.path.join(ckpt_dir, f"best_{tag}.pth"))

    def _persist_best_for_resume(path, enc, tft):
        """Stable, best-known checkpoint used by resume logic."""
        payload = {'tft_state_dict': {k: v.detach().cpu() for k, v in tft.state_dict().items()}}
        if FINE_TUNE_ENCODER:
            payload['encoder_state_dict'] = {k: v.detach().cpu()
                                             for k, v in enc.state_dict().items()}
        torch.save(payload, path)

    def _load_best(path):
        """Load (enc, tft) from a resume checkpoint. Encoder is optional when frozen."""
        sw = torch.load(path, map_location='cpu')
        enc = fresh_encoder()
        if FINE_TUNE_ENCODER and 'encoder_state_dict' in sw:
            enc.load_state_dict(sw['encoder_state_dict'])
        tft = build_tft()
        tft.load_state_dict(sw['tft_state_dict'])
        return enc, tft

    import re as _re

    def _find_latest_partial(ckpt_dir, tag):
        if not os.path.isdir(ckpt_dir):
            return None, 0, 0
        pat = _re.compile(rf"^partial_{_re.escape(tag)}_e(\d+)_s(\d+)\.pth$")
        best = (None, 0, 0)
        for fname in os.listdir(ckpt_dir):
            m = pat.match(fname)
            if not m:
                continue
            e, s = int(m.group(1)), int(m.group(2))
            if (e, s) > (best[1], best[2]):
                best = (os.path.join(ckpt_dir, fname), e, s)
        return best





    def train_epoch(enc, tft, data, opt, crit, scaler,
                    epoch=None, ckpt_dir=None, tag="run",
                    save_every=None, do_empty_cache=False):
        if FINE_TUNE_ENCODER: enc.train()
        else: enc.eval()
        tft.train()

        if save_every is None:
            save_every = CKPT_EVERY_N_STEPS

        tot_t       = torch.zeros((), device=DEVICE)
        tot_bin_t   = 0.0
        tot_multi_t = 0.0
        gn_t        = torch.zeros((), device=DEVICE)
        n = 0
        gn_batches = 0
        opt.zero_grad(set_to_none=True)

        pbar = tqdm(enumerate(data),
                    total=len(data),
                    desc=f"epoch {epoch}" if epoch is not None else tag,
                    dynamic_ncols=True,
                    leave=False)
        for step, item in pbar:
            if step % 5 == 0:
                torch.cuda.empty_cache()
            if cache_train is not None:
                emb, struct_seq, y_bin, y_multi = item
                emb = emb.to(DEVICE, non_blocking=True).float()
            else:
                flat_batch, B, T, struct_seq, y_bin, y_multi = item

            y_bin = y_bin.to(DEVICE); y_multi = y_multi.to(DEVICE)
            struct_seq = struct_seq.to(DEVICE, non_blocking=True)

            with autocast_ctx():
                if cache_train is None:
                    graph_emb = _encode_sequence(enc, flat_batch, B, T, DEVICE,
                                                frozen=not FINE_TUNE_ENCODER)
                else:
                    graph_emb = emb
                bin_logits, multi_logits, _, _ = tft(graph_emb.float(), struct_seq)
                loss, l_bin, l_multi = crit(bin_logits, multi_logits, y_bin, y_multi)
                loss = loss / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()

            if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(data):
                scaler.unscale_(opt)
                params = list(tft.parameters())
                if FINE_TUNE_ENCODER: params = list(enc.parameters()) + params
                gn = torch.nn.utils.clip_grad_norm_(params, 1.0)
                gn_t += gn; gn_batches += 1
                scaler.step(opt); scaler.update()
                opt.zero_grad(set_to_none=True)

            tot_t       += loss.detach() * GRAD_ACCUM_STEPS
            tot_bin_t   += l_bin if torch.is_tensor(l_bin) else float(l_bin)
            tot_multi_t += l_multi if torch.is_tensor(l_multi) else float(l_multi)
            n += 1

            pbar.set_postfix({
                'loss':  f"{float(tot_t) / max(n, 1):.4f}",
                'bin':   f"{tot_bin_t / max(n, 1):.4f}",
                'multi': f"{tot_multi_t / max(n, 1):.4f}",
                'gnorm': f"{float(gn_t) / max(gn_batches, 1):.3f}",
            })

            if ckpt_dir is not None and save_every > 0 and (step + 1) % save_every == 0:
                _save_partial_light(ckpt_dir, tag,
                                    epoch if epoch is not None else -1, step + 1,
                                    enc, tft)

        pbar.close()

        denom = max(n, 1)
        return {
            'total':  float(tot_t) / denom,
            'binary': tot_bin_t / denom,
            'multi':  tot_multi_t / denom,
            'gnorm':  float(gn_t) / max(gn_batches, 1),
        }

    @torch.inference_mode()
    def evaluate(enc, tft, data, collect_attn=False):
        enc.eval(); tft.eval()
        bp, mp, yb, ym = [], [], [], []
        attn_last_all = []
        for item in data:
            if cache_train is not None or (data is cache_val) or (data is cache_test):
                emb, struct_seq, y_bin, y_multi = item
                emb = emb.to(DEVICE, non_blocking=True).float()
            else:
                flat_batch, B, T, struct_seq, y_bin, y_multi = item

            struct_seq = struct_seq.to(DEVICE, non_blocking=True)
            if cache_train is not None or (data is cache_val) or (data is cache_test):
                graph_emb = emb
            else:
                graph_emb = _encode_sequence(enc, flat_batch, B, T, DEVICE, frozen=True)
            bin_logits, multi_logits, attn_w, _ = tft(graph_emb.float(), struct_seq)
            bp.append(torch.sigmoid(bin_logits.float()).cpu().numpy())
            mp.append(F.softmax(multi_logits.float(), -1).cpu().numpy())
            yb.append(y_bin.numpy()); ym.append(y_multi.numpy())
            if collect_attn:
                attn_last_all.append(attn_w.float()[:, -1, :].cpu().numpy())
        out = (np.concatenate(bp), np.concatenate(mp),
               np.concatenate(yb), np.concatenate(ym))
        if collect_attn:
            out = out + (np.concatenate(attn_last_all),)
        return out

    def train_one_seed(seed, verbose=False, tag="", ckpt_dir=None,
                       selection_metric='macro_f1_merged',
                       warmup_epochs=None, patience=None,
                       warm_start=False):
        if warmup_epochs is None: warmup_epochs = WARMUP_EPOCHS
        if patience is None:      patience = PATIENCE

        torch.manual_seed(seed); np.random.seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

        enc = fresh_encoder(); tft = build_tft()
        opt = make_optimizer(enc, tft)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
        sc = GradScaler()

        best_score = -1.0
        best_epoch = 0
        best_enc   = None
        best_tft   = None
        no_imp     = 0
        start_epoch = 1

        safe_tag = (tag or f"seed{seed}").replace("/", "_").replace(" ", "_")

        # ---- warm-start: prefer latest partial, fall back to best ----
        if warm_start and ckpt_dir is not None:
            best_path = os.path.join(ckpt_dir, f"best_{safe_tag}.pth")
            part_path, part_ep, part_step = _find_latest_partial(ckpt_dir, safe_tag)

            # --- 1) read reference best (for best_score / fallback weights) ---
            ref_best_score  = -1.0
            ref_best_epoch  = 0
            ref_best_enc_sd = None
            ref_best_tft_sd = None
            if os.path.isfile(best_path):
                sb = torch.load(best_path, map_location=DEVICE)
                ref_best_epoch  = int(sb.get('epoch', 0))
                ref_best_score  = float(sb.get('metrics', {})
                                        .get(f'val_{selection_metric}', -1.0))
                if FINE_TUNE_ENCODER and 'encoder_state_dict' in sb:
                    ref_best_enc_sd = sb['encoder_state_dict']
                ref_best_tft_sd = sb['tft_state_dict']

            # --- 2) pick weights for the working model ---
            loaded_from = None
            if part_path is not None:
                sp = torch.load(part_path, map_location=DEVICE)
                if FINE_TUNE_ENCODER and 'encoder_state_dict' in sp:
                    enc.load_state_dict(sp['encoder_state_dict'])
                tft.load_state_dict(sp['tft_state_dict'])
                start_epoch = int(part_ep) + 1
                loaded_from = (f"partial {os.path.basename(part_path)} "
                               f"(ep {part_ep}, step {part_step})  >  start ep {start_epoch}")
            elif ref_best_tft_sd is not None:
                if FINE_TUNE_ENCODER and ref_best_enc_sd is not None:
                    enc.load_state_dict(ref_best_enc_sd)
                tft.load_state_dict(ref_best_tft_sd)
                start_epoch = ref_best_epoch + 1
                loaded_from = (f"best (ep {ref_best_epoch})  >  start ep {start_epoch}")
            # else: nothing on disk  >  normal cold start (start_epoch stays 1)

            if loaded_from is not None:
                # advance cosine LR to the epoch we're resuming at
                for _ in range(start_epoch - 1):
                    sched.step()

                # --- 3) restore best trackers ---
                if ref_best_tft_sd is not None:
                    # keep the peak-val model as the fallback reference
                    if FINE_TUNE_ENCODER and ref_best_enc_sd is not None:
                        best_enc = {k: v.clone() for k, v in ref_best_enc_sd.items()}
                    else:
                        best_enc = {k: v.clone() for k, v in enc.state_dict().items()}
                    best_tft = {k: v.clone() for k, v in ref_best_tft_sd.items()}
                    best_score = ref_best_score
                    best_epoch = ref_best_epoch
                else:
                    # partial only, no best.pth: seed trackers with the loaded weights
                    best_enc = {k: v.clone() for k, v in enc.state_dict().items()}
                    best_tft = {k: v.clone() for k, v in tft.state_dict().items()}
                    best_score = -1.0
                    best_epoch = 0

                no_imp = 0
                if verbose:
                    print(f"  [warm-start] {loaded_from}")
                    print(f"  [warm-start] reference best: "
                          f"{selection_metric}={best_score:.4f} @ ep {best_epoch} "
                          f"(Adam/scheduler-schedule/RNG NOT restored)")

        if verbose: _print_epoch_header(tag, EPOCHS)
        t0 = time.time()

        for ep in range(start_epoch, EPOCHS + 1):
            torch.cuda.empty_cache()
            ep_t0 = time.time()
            tr = train_epoch(enc, tft, _train_data(shuffle=True), opt, criterion, sc,
                             epoch=ep, ckpt_dir=ckpt_dir, tag=safe_tag,
                             save_every=CKPT_EVERY_N_STEPS)

            pb_val, pm_val, yb_val, ym_val = evaluate(enc, tft, _val_data())
            val = _val_metrics(pb_val, pm_val, yb_val, ym_val,
                               thr=0.5,
                               train_visible_idx=train_visible_idx,
                               class_names=class_names,
                               dataset=DATASET_NAME)

            sched.step()
            current_lr = opt.param_groups[-1]['lr']
            epoch_time = time.time() - ep_t0

            score     = val[selection_metric]
            warmed_up = ep >= warmup_epochs
            improved  = warmed_up and (score > best_score + 1e-4)

            if improved:
                best_score = score
                best_epoch = ep
                best_enc = {k: v.clone() for k, v in enc.state_dict().items()}
                best_tft = {k: v.clone() for k, v in tft.state_dict().items()}
                no_imp = 0
                if ckpt_dir is not None:
                    _save_best_light(ckpt_dir, safe_tag, ep, enc, tft,
                        {f'val_{selection_metric}': float(score),
                         'val_macro_f1_merged':  float(val['macro_f1_merged']),
                         'val_roc':              float(val['roc']),
                         'val_f1_binary':        float(val['f1']),
                         'selection_metric':     selection_metric})
            else:
                if warmed_up:
                    no_imp += 1

            patience_left = max(patience - no_imp, 0)
            note = "★ best" if improved else ("  (warmup)" if not warmed_up else "")

            if verbose:
                _print_epoch_row(ep, EPOCHS, tr, val, tr['gnorm'], current_lr,
                                 patience_left, best_score, epoch_time, note)

            if warmed_up and no_imp >= patience:
                if verbose:
                    print(f"  ↳ early stop at epoch {ep} "
                          f"(no {selection_metric} improvement for {patience} epochs)")
                break

        if verbose:
            total_time = time.time() - t0
            print(f"  ↳ done: best val {selection_metric} {best_score:.4f} "
                  f"@ epoch {best_epoch}  | total {_fmt_time(total_time)}")

        if best_enc is None:
            best_enc = {k: v.clone() for k, v in enc.state_dict().items()}
            best_tft = {k: v.clone() for k, v in tft.state_dict().items()}
            best_score = float(val[selection_metric]) if 'val' in locals() else 0.0

        enc.load_state_dict(best_enc); tft.load_state_dict(best_tft)
        return enc, tft, best_score

    # MAIN SINGLE-SEED
    if main_done:
        print("\n" + "=" * 70)
        print(f"[resume] loading main seed from {BEST_MAIN}")
        print("=" * 70)
        main_enc, main_tft = _load_best(BEST_MAIN)
        main_enc.eval(); main_tft.eval()

        pb_test, pm_test, yb_test, ym_test, attn_test = evaluate(
            main_enc, main_tft, _test_data(), collect_attn=True)
        pb_val, pm_val, yb_val, ym_val = evaluate(main_enc, main_tft, _val_data())

        with open(MAIN_JSON) as f:
            _mj = json.load(f)
        main_val_score = float(_mj.get('val_selection_score', 0.0))
        print(f"[resume] main seed val selection score = {main_val_score:.4f}")
    else:
        _main_warm = resume_from_partials and not force_retrain and (
            os.path.isfile(BEST_MAIN)
            or _find_latest_partial(PARTIAL_DIR, f"{DATASET_NAME}_{SSL_METHOD}")[0] is not None
        )
        if _main_warm:
            print("\n" + "=" * 70)
            print(f"[warm-start] main seed: found partials/best — resuming from last checkpoint")
            print("=" * 70)
        else:
            print("\n" + "=" * 70)
            print(f"SINGLE-SEED TRAINING (seed={SEED}, SSL={SSL_METHOD.upper()})")
            print("=" * 70)
        main_enc, main_tft, main_val_score = train_one_seed(
            SEED, verbose=True,
            tag=f"{DATASET_NAME}_{SSL_METHOD}",
            ckpt_dir=PARTIAL_DIR,
            selection_metric=MAIN_SELECTION_METRIC,
            warm_start=_main_warm)

        pb_test, pm_test, yb_test, ym_test, attn_test = evaluate(
            main_enc, main_tft, _test_data(), collect_attn=True)
        pb_val, pm_val, yb_val, ym_val = evaluate(main_enc, main_tft, _val_data())

        _persist_best_for_resume(BEST_MAIN, main_enc, main_tft)
        print(f"[resume] main seed checkpoint written  >  {BEST_MAIN}")

    yb_val_i  = yb_val.astype(int)
    yb_test_i = yb_test.astype(int)

    thr_f1opt, _ = _optimal_threshold(pb_val, yb_val_i)
    thr_prior    = _prior_matched_threshold(pb_val, yb_val_i)
    thr_headline = thr_prior if USE_PRIOR_MATCHED_THRESHOLD else thr_f1opt

    m_f1opt = _binary_metrics_at(pb_test, yb_test_i, thr_f1opt)
    m_prior = _binary_metrics_at(pb_test, yb_test_i, thr_prior)
    m_head  = m_prior if USE_PRIOR_MATCHED_THRESHOLD else m_f1opt
    bin_roc = float(roc_auc_score(yb_test_i, pb_test))
    bin_pr  = float(average_precision_score(yb_test_i, pb_test))
    m_05 = _binary_metrics_at(pb_test, yb_test_i, 0.5)
    print(f"  BINARY @ 0.5 fixed  : ROC {bin_roc:.4f}  PR {bin_pr:.4f}  "
          f"F1 {m_05['f1']:.4f}  FPR {m_05['fpr']:.4f}  FNR {m_05['fnr']:.4f}  "
          f"pred_attack_rate {m_05['pred_attack_rate']:.3f}")

    ym_pred = pm_test.argmax(-1)
    multi_mf1_raw   = float(f1_score(ym_test, ym_pred, average='macro',    zero_division=0))
    multi_wf1       = float(f1_score(ym_test, ym_pred, average='weighted', zero_division=0))
    mask_te_vis = np.isin(ym_test, train_visible_idx)
    if mask_te_vis.sum() >= 2 and len(np.unique(ym_test[mask_te_vis])) >= 2:
        multi_mf1_visible = float(f1_score(ym_test[mask_te_vis], ym_pred[mask_te_vis],
                                           average='macro', zero_division=0))
    else:
        multi_mf1_visible = multi_mf1_raw
    ym_test_m = np.array([map_to_family(class_names[i], DATASET_NAME) for i in ym_test])
    ym_pred_m = np.array([map_to_family(class_names[i], DATASET_NAME) for i in ym_pred])
    mask_m_te = np.isin(ym_test_m, train_visible_fams)
    
    multi_mf1_merged = multi_mf1_raw

    print(f"\nMAIN TEST ({SSL_METHOD.upper()}, seed={SEED})")
    print(f"  BINARY threshold     : headline={thr_headline:.3f}  "
          f"(f1opt={thr_f1opt:.3f}, prior={thr_prior:.3f})")
    print(f"  BINARY @ headline    : ROC {bin_roc:.4f}  PR {bin_pr:.4f}  "
          f"F1 {m_head['f1']:.4f}  FPR {m_head['fpr']:.4f}  FNR {m_head['fnr']:.4f}  "
          f"pred_attack_rate {m_head['pred_attack_rate']:.3f}")
    print(f"  BINARY @ F1-opt      : F1 {m_f1opt['f1']:.4f}  "
          f"FPR {m_f1opt['fpr']:.4f}  FNR {m_f1opt['fnr']:.4f}  "
          f"pred_attack_rate {m_f1opt['pred_attack_rate']:.3f}")
    print(f"  MULTI (raw)          : macro-F1 {multi_mf1_raw:.4f}  weighted-F1 {multi_wf1:.4f}")
    print(f"  MULTI (fine, vis)    : macro-F1 {multi_mf1_visible:.4f}")
    print(f"  MULTI (merged fam)   : macro-F1 {multi_mf1_merged:.4f}")

    per_class_rows = _per_class_score_analysis(pb_test, test_labels_seq,
                                               class_names, thr_headline)
    _print_per_class_table(per_class_rows, thr_headline, f"test, seed={SEED}")

    _flush_report(PHASE3_DIR, "phase3_main_seed_report.json", {
        'stage': 'single_seed',
        'dataset': DATASET_NAME, 'ssl_method': SSL_METHOD,
        'seed': SEED,
        'val_selection_score': float(main_val_score),
        'selection_metric': 'val_macro_f1_merged',
        'thresholds': {
            'headline':    float(thr_headline),
            'f1_optimal':  float(thr_f1opt),
            'prior_match': float(thr_prior),
            'protocol':    'prior_matched' if USE_PRIOR_MATCHED_THRESHOLD else 'f1_optimal',
        },
        'test': {
            'binary_roc':    float(bin_roc),
            'binary_pr':     float(bin_pr),
            'binary_f1':     float(m_head['f1']),
            'binary_fpr':    float(m_head['fpr']),
            'binary_fnr':    float(m_head['fnr']),
            'binary_f1_at_f1opt':   float(m_f1opt['f1']),
            'binary_fpr_at_f1opt':  float(m_f1opt['fpr']),
            'binary_fnr_at_f1opt':  float(m_f1opt['fnr']),
            'pred_attack_rate':     float(m_head['pred_attack_rate']),
            'true_attack_rate':     float(yb_test_i.mean()),
            'multiclass_mf1_raw':     float(multi_mf1_raw),
            'multiclass_mf1_visible': float(multi_mf1_visible),
            'multiclass_mf1_merged':  float(multi_mf1_merged),
            'multiclass_wf1':         float(multi_wf1),
        },
        'per_class_seed42':  per_class_rows,
        'train_visible_fams': train_visible_fams,
    })

    # attention maps (always recomputed from attn_test, which is loaded or fresh)
    print(f"\nSaving attention maps to {ATTN_DIR} ...")
    mean_attn = attn_test.mean(axis=0)
    per_class_attn = np.zeros((NUM_CLASSES, SEQUENCE_LENGTH), dtype=np.float32)
    class_counts = np.zeros(NUM_CLASSES, dtype=int)
    for cls_i in range(NUM_CLASSES):
        mask = (ym_test == cls_i)
        class_counts[cls_i] = int(mask.sum())
        if mask.any():
            per_class_attn[cls_i] = attn_test[mask].mean(axis=0)
    np.save(os.path.join(ATTN_DIR, "mean_attention.npy"), mean_attn)
    np.save(os.path.join(ATTN_DIR, "per_class_attention.npy"), per_class_attn)
    np.save(os.path.join(ATTN_DIR, "all_test_attn.npy"), attn_test)
    with open(os.path.join(ATTN_DIR, "class_names.json"), 'w') as f:
        json.dump({'class_names': class_names,
                   'class_counts': class_counts.tolist(),
                   'sequence_length': SEQUENCE_LENGTH}, f, indent=2)

    # FEW-SHOT SWEEP  (resume-aware: each frac is a JSON marker)
    print("\n" + "=" * 70)
    print("FEW-SHOT SWEEP")
    print("=" * 70)
    rng_split = np.random.default_rng(SEED)
    perm = rng_split.permutation(len(train_starts))
    fewshot_results = {}

    for frac in FRACTIONS:
        fs_json = _fewshot_json(frac)
        if resume_from_partials and not force_retrain and os.path.isfile(fs_json):
            with open(fs_json) as f:
                loaded = json.load(f)
            for k in ('stage', 'dataset', 'ssl_method', 'frac'):
                loaded.pop(k, None)
            fewshot_results[frac] = loaded
            print(f"\n  ── [resume] few-shot frac={frac:.0%} loaded from {fs_json} ──")
            print(f"  ↳ frac {frac:.0%}: test ROC {loaded.get('test_roc', float('nan')):.4f} | "
                  f"test F1 {loaded.get('test_f1', float('nan')):.4f}")
            continue

        n_use = max(20, int(frac * len(train_starts)))
        idx = perm[:n_use]
        sub_starts = train_starts[idx]
        sub_y_bin  = train_y_bin[idx]; sub_y_multi = train_y_multi[idx]

        if cache_train is not None:
            n_batches = max(1, int(np.ceil(n_use / BATCH_SIZE)))
            sub_data = [cache_train[i] for i in rng_split.permutation(len(cache_train))[:n_batches]]
            sub_len = len(sub_data)
        else:
            sub_loader = make_loader(train_graphs, sub_starts, sub_y_bin, sub_y_multi,
                                     str_train, True)
            sub_data, sub_len = sub_loader, len(sub_loader)

        sel_metric   = 'roc'
        fu_warmup    = FEWSHOT_WARMUP_SMALL if n_use < SMALL_LABEL_THRESHOLD else FEWSHOT_WARMUP
        fu_patience  = FEWSHOT_PATIENCE_SMALL if n_use < SMALL_LABEL_THRESHOLD else PATIENCE

        print(f"\n  ── few-shot frac={frac:.0%} (n={n_use}) "
              f"selection={sel_metric} warmup={fu_warmup} patience={fu_patience} ──")

        torch.manual_seed(SEED); np.random.seed(SEED)
        enc = fresh_encoder(); tft = build_tft()
        opt2 = make_optimizer(enc, tft)
        sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt2, T_max=FEWSHOT_EPOCHS, eta_min=1e-6)
        sc2 = GradScaler()

        fs_tag = f"{DATASET_NAME}_{SSL_METHOD}_fewshot{int(frac*100):03d}"
        b_score = -1.0; b_enc = None; b_tft = None; best_ep = 0
        no_imp = 0
        _print_epoch_header(f"fewshot-{frac:.0%}", FEWSHOT_EPOCHS)
        t0 = time.time()

        for ep in range(1, FEWSHOT_EPOCHS + 1):
            ep_t0 = time.time()
            data_ep = ([sub_data[i] for i in np.random.permutation(len(sub_data))]
                       if cache_train is not None else sub_data)
            tr = train_epoch(enc, tft, data_ep, opt2, criterion, sc2,
                             epoch=ep, ckpt_dir=PARTIAL_DIR, tag=fs_tag,
                             save_every=CKPT_EVERY_N_STEPS)
            pbv, pmv, ybv, ymv = evaluate(enc, tft, _val_data())
            val = _val_metrics(pbv, pmv, ybv, ymv,
                               thr=0.5,
                               train_visible_idx=train_visible_idx,
                               class_names=class_names,
                               dataset=DATASET_NAME)
            sched2.step()
            cur_lr = opt2.param_groups[-1]['lr']
            dt = time.time() - ep_t0

            score     = val[sel_metric]
            warmed_up = ep >= fu_warmup
            improved  = warmed_up and (score > b_score + 1e-4)

            if improved:
                b_score = score; best_ep = ep
                b_enc = {k: v.clone() for k, v in enc.state_dict().items()}
                b_tft = {k: v.clone() for k, v in tft.state_dict().items()}
                no_imp = 0
                _save_best_light(PARTIAL_DIR, fs_tag, ep, enc, tft,
                    {f'val_{sel_metric}': float(score),
                     'val_macro_f1_merged': float(val['macro_f1_merged']),
                     'val_roc':             float(val['roc']),
                     'frac':                float(frac),
                     'selection_metric':    sel_metric})
            else:
                if warmed_up:
                    no_imp += 1

            patience_left = max(fu_patience - no_imp, 0)
            note = "★ best" if improved else ("  (warmup)" if not warmed_up else "")
            _print_epoch_row(ep, FEWSHOT_EPOCHS, tr, val, tr['gnorm'], cur_lr,
                             patience_left, b_score, dt, note)

            if warmed_up and no_imp >= fu_patience:
                print(f"  ↳ early stop at epoch {ep} "
                      f"(no val_{sel_metric} improvement for {fu_patience} epochs)")
                break

        if b_enc is None:
            b_enc = {k: v.clone() for k, v in enc.state_dict().items()}
            b_tft = {k: v.clone() for k, v in tft.state_dict().items()}
            b_score = float(val[sel_metric])

        enc.load_state_dict(b_enc); tft.load_state_dict(b_tft)
        pbt, pmt, ybt, ymt = evaluate(enc, tft, _test_data())
        ybt_i = ybt.astype(int)
        troc = float(roc_auc_score(ybt_i, pbt))
        thr_fs = _prior_matched_threshold(pbt, ybt_i) if USE_PRIOR_MATCHED_THRESHOLD \
                 else _optimal_threshold(pbt, ybt_i)[0]
        m_fs = _binary_metrics_at(pbt, ybt_i, thr_fs)
        ymt_pred = pmt.argmax(-1)
        ymt_m      = np.array([map_to_family(class_names[i], DATASET_NAME) for i in ymt])
        ymt_pred_m = np.array([map_to_family(class_names[i], DATASET_NAME) for i in ymt_pred])
        mask_m_t = np.isin(ymt_m, train_visible_fams)
        tmf1_merged = tmf1_raw
        tmf1_raw = float(f1_score(ymt, ymt_pred, average='macro', zero_division=0))

        fewshot_results[frac] = {
            'n_train':               int(n_use),
            'selection_metric':      sel_metric,
            'val_selection_score':   float(b_score),
            'test_f1':               float(m_fs['f1']),
            'test_f1_at_0.5':        float(f1_score(ybt_i, (pbt >= 0.5).astype(int),
                                                    average='binary', zero_division=0)),
            'test_roc':              float(troc),
            'test_fpr':              float(m_fs['fpr']),
            'test_fnr':              float(m_fs['fnr']),
            'test_threshold':        float(thr_fs),
            'test_macro_f1_merged':  float(tmf1_merged),
            'test_macro_f1_raw':     float(tmf1_raw),
            'best_epoch':            int(best_ep),
        }
        print(f"  ↳ frac {frac:.0%}: val sel({sel_metric}) {b_score:.4f} (ep {best_ep}) | "
              f"test ROC {troc:.4f} | test F1 {m_fs['f1']:.4f} @ thr {thr_fs:.2f} | "
              f"mF1(merged) {tmf1_merged:.4f} | total {_fmt_time(time.time()-t0)}")

        _flush_report(PHASE3_DIR, f"phase3_fewshot_{int(frac*100):03d}.json", {
            'stage': 'fewshot',
            'dataset': DATASET_NAME, 'ssl_method': SSL_METHOD,
            'frac': float(frac),
            **fewshot_results[frac],
        })

        del enc, tft, opt2, sched2, sc2, sub_data
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()

    fig, ax = plt.subplots(figsize=(7, 4.5))
    xs = [f*100 for f in FRACTIONS]
    ax.plot(xs, [fewshot_results[f]['test_roc']             for f in FRACTIONS], marker='^', lw=1.8, label='Test ROC')
    ax.plot(xs, [fewshot_results[f]['test_f1']              for f in FRACTIONS], marker='o', lw=1.8, label='Test F1 (prior-matched)')
    ax.plot(xs, [fewshot_results[f]['test_macro_f1_merged'] for f in FRACTIONS], marker='s', lw=1.8, label='Test macro-F1 (merged)', linestyle='--')
    ax.set_xlabel("% labelled train seqs"); ax.set_ylabel("Metric")
    ax.set_title(f"MC-STGAT-IDS · {DATASET_NAME} · {SSL_METHOD.upper()} · label efficiency")
    ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout()
    fig.savefig(os.path.join(PHASE3_DIR, "fig_label_efficiency.png"), dpi=150, bbox_inches='tight')
    fig.savefig(os.path.join(PHASE3_DIR, "fig_label_efficiency.pdf"), bbox_inches='tight')
    plt.close(fig)

    print("\n" + "=" * 70)
    print(f"MULTI-SEED ENSEMBLE ({len(ensemble_seeds)} seeds: {ensemble_seeds})")
    print("=" * 70)
    ens_bin_probs = None; ens_multi_probs = None
    ens_y_bin = None; ens_y_multi = None; ens_attn = None
    ens_val_probs = None
    seed_results = []

    ens_ym_m_pre = np.array([map_to_family(class_names[i], DATASET_NAME) for i in test_y_multi])
    assert len(ens_ym_m_pre) == len(test_starts)

    for i, s in enumerate(ensemble_seeds):
        seed_json = _ens_seed_json(s)
        seed_ckpt = _ens_seed_ckpt(s)
        seed_tag  = f"{DATASET_NAME}_{SSL_METHOD}_seed{s}"
        seed_partial, seed_p_ep, seed_p_step = _find_latest_partial(PARTIAL_DIR, seed_tag)

        resume_this_seed = (resume_from_partials and not force_retrain
                            and os.path.isfile(seed_json)
                            and os.path.isfile(seed_ckpt))
        warm_this_seed   = (not resume_this_seed
                            and resume_from_partials and not force_retrain
                            and (os.path.isfile(seed_ckpt) or seed_partial is not None))

        tag_state = ("[resume: load + re-evaluate]" if resume_this_seed
                     else (f"[warm-start from {os.path.basename(seed_partial)}]"
                           if (warm_this_seed and seed_partial is not None)
                           else ("[warm-start from best ckpt]" if warm_this_seed
                                 else "")))
        print(f"\n{'='*70}")
        print(f"ENSEMBLE SEED {s}  ({i+1}/{len(ensemble_seeds)})  {tag_state}")
        print(f"{'='*70}")

        if resume_this_seed:
            enc_s, tft_s = _load_best(seed_ckpt)
            pbt, pmt, ybt, ymt, attn_s = evaluate(enc_s, tft_s, _test_data(),
                                                  collect_attn=True)
            pbv_s, _, ybv_s, _ = evaluate(enc_s, tft_s, _val_data())
            with open(seed_json) as f:
                _sj = json.load(f)
            vscore = float(_sj['val_selection_score'])
            troc   = float(_sj['test_roc'])
            smf1   = float(_sj['test_macro_f1_merged'])
            del enc_s, tft_s
            gc.collect()
            if DEVICE.type == 'cuda': torch.cuda.empty_cache()
        else:
            enc_s, tft_s, vscore = train_one_seed(
                s, verbose=ENSEMBLE_VERBOSE,
                tag=seed_tag,
                ckpt_dir=PARTIAL_DIR,
                selection_metric=MAIN_SELECTION_METRIC,
                warm_start=warm_this_seed)
            pbt, pmt, ybt, ymt, attn_s = evaluate(enc_s, tft_s, _test_data(),
                                                  collect_attn=True)
            pbv_s, _, ybv_s, _ = evaluate(enc_s, tft_s, _val_data())
            ybt_i = ybt.astype(int)
            troc  = float(roc_auc_score(ybt_i, pbt))
            ymt_pred = pmt.argmax(-1)
            ymt_m      = ens_ym_m_pre
            ymt_pred_m = np.array([map_to_family(class_names[j], DATASET_NAME) for j in ymt_pred])
            mask_m = np.isin(ymt_m, train_visible_fams)
            if mask_m.sum() >= 2 and len(np.unique(ymt_m[mask_m])) >= 2:
                smf1 = float(f1_score(ymt_m[mask_m], ymt_pred_m[mask_m],
                                      average='macro', zero_division=0))
            else:
                smf1 = 0.0

            _persist_best_for_resume(seed_ckpt, enc_s, tft_s)
            _flush_report(PHASE3_DIR, f"phase3_ensemble_seed{s}.json", {
                'stage': 'ensemble_seed',
                'dataset': DATASET_NAME, 'ssl_method': SSL_METHOD,
                'seed': int(s), 'val_selection_score': float(vscore),
                'test_roc': float(troc), 'test_macro_f1_merged': float(smf1),
            })
            del enc_s, tft_s
            gc.collect()
            if DEVICE.type == 'cuda': torch.cuda.empty_cache()

        seed_results.append({
            'seed': int(s), 'val_selection_score': float(vscore),
            'test_roc': float(troc), 'test_macro_f1_merged': float(smf1),
        })
        print(f"  seed {s}: val merged-mF1 {vscore:.4f} | test ROC {troc:.4f} | "
              f"test merged-mF1 {smf1:.4f}")

        ens_bin_probs   = pbt if ens_bin_probs is None else ens_bin_probs + pbt
        ens_multi_probs = pmt if ens_multi_probs is None else ens_multi_probs + pmt
        ens_attn        = attn_s if ens_attn is None else ens_attn + attn_s
        ens_val_probs   = pbv_s if ens_val_probs is None else ens_val_probs + pbv_s
        ens_y_bin = ybt; ens_y_multi = ymt

    ens_bin_probs   /= len(ensemble_seeds)
    ens_multi_probs /= len(ensemble_seeds)
    ens_attn        /= len(ensemble_seeds)
    ens_val_probs   /= len(ensemble_seeds)

    thr_f1opt_ens = _optimal_threshold(ens_val_probs, yb_val_i)[0]
    thr_prior_ens = _prior_matched_threshold(ens_val_probs, yb_val_i)
    thr_ens       = thr_prior_ens if USE_PRIOR_MATCHED_THRESHOLD else thr_f1opt_ens
    print(f"Ensemble threshold : headline={thr_ens:.3f}  "
          f"(f1opt={thr_f1opt_ens:.3f}, prior={thr_prior_ens:.3f})")

    np.save(os.path.join(ATTN_DIR, "ensemble_mean_attention.npy"), ens_attn.mean(axis=0))
    np.save(os.path.join(ATTN_DIR, "ensemble_all_test_attn.npy"), ens_attn)

    ens_y_bin_i = ens_y_bin.astype(int)
    ens_roc = float(roc_auc_score(ens_y_bin_i, ens_bin_probs))
    ens_pr  = float(average_precision_score(ens_y_bin_i, ens_bin_probs))
    m_ens   = _binary_metrics_at(ens_bin_probs, ens_y_bin_i, thr_ens)

    ens_multi_pred = ens_multi_probs.argmax(-1)
    ens_ym_m       = ens_ym_m_pre
    ens_ypred_m    = np.array([map_to_family(class_names[i], DATASET_NAME) for i in ens_multi_pred])
    mask_ens       = np.isin(ens_ym_m, train_visible_fams)
    
    ens_multi_mf1_merged = ens_multi_mf1_raw
    
    ens_multi_mf1_raw = float(f1_score(ens_y_multi, ens_multi_pred,
                                       average='macro', zero_division=0))
    ens_multi_wf1 = float(f1_score(ens_y_multi, ens_multi_pred,
                                   average='weighted', zero_division=0))

    print(f"\nENSEMBLE — ROC {ens_roc:.4f}  PR {ens_pr:.4f}  "
          f"F1 {m_ens['f1']:.4f}  FPR {m_ens['fpr']:.4f}  FNR {m_ens['fnr']:.4f}  "
          f"pred_attack_rate {m_ens['pred_attack_rate']:.3f}  (thr={thr_ens:.2f})")
    print(f"ENSEMBLE — macro-F1 (merged) {ens_multi_mf1_merged:.4f}  "
          f"macro-F1 (raw) {ens_multi_mf1_raw:.4f}  weighted-F1 {ens_multi_wf1:.4f}")

    per_class_rows_ens = _per_class_score_analysis(ens_bin_probs, test_labels_seq,
                                                   class_names, thr_ens)
    _print_per_class_table(per_class_rows_ens, thr_ens, "test, ensemble")

    rng_b = np.random.default_rng(BOOTSTRAP_SEED)
    n = len(ens_y_bin_i); boot = np.empty(BOOTSTRAP_N)
    for b in range(BOOTSTRAP_N):
        idx = rng_b.integers(0, n, size=n)
        boot[b] = roc_auc_score(ens_y_bin_i[idx], ens_bin_probs[idx])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"\nEnsemble ROC-AUC = {ens_roc:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

    torch.save({
        'encoder_state_dict': main_enc.state_dict(),
        'tft_state_dict': main_tft.state_dict(),
        'class_names': class_names, 'label_to_idx': label_to_idx,
        'train_visible_fams': train_visible_fams,
        'sequence_length': SEQUENCE_LENGTH, 'num_struct_feats': NUM_STRUCT_FEATS,
        'num_classes': NUM_CLASSES, 'hidden_dim': HIDDEN_DIM,
        'tft_hidden_dim': TFT_HIDDEN_DIM,
        'ssl_method': SSL_METHOD, 'ssl_objective': ssl_objective,
        'fine_tuned_encoder': bool(FINE_TUNE_ENCODER),
        'optimal_threshold':          float(thr_headline),
        'ensemble_optimal_threshold': float(thr_ens),
        'threshold_protocol': 'prior_matched' if USE_PRIOR_MATCHED_THRESHOLD else 'f1_optimal',
        'binary_test_roc': float(bin_roc), 'binary_test_f1': float(m_head['f1']),
        'multiclass_test_macro_f1_merged': float(multi_mf1_merged),
        'max_window_gap_sec': float(MAX_WINDOW_GAP_SEC),
    }, CKPT_FILE)

    report = {
        'phase': 3, 'dataset': DATASET_NAME, 'window_sec': 10,
        'task': 'binary_primary_multiclass_secondary',
        'ssl_method': SSL_METHOD, 'ssl_objective': ssl_objective,
        'fine_tuned_encoder': bool(FINE_TUNE_ENCODER),
        'attack_threshold': attack_threshold,
        'max_window_gap_sec': float(MAX_WINDOW_GAP_SEC),
        'gap_enforcement': bool(ENFORCE_MAX_WINDOW_GAP),
        'selection_criterion': 'val_macro_f1_merged',
        'threshold_protocol': 'prior_matched' if USE_PRIOR_MATCHED_THRESHOLD else 'f1_optimal',
        'use_balanced_sampler': bool(use_balanced_sampler),
        'warmup_epochs': WARMUP_EPOCHS,
        'train_visible_fams': train_visible_fams,
        'main_run': {
            'seed': SEED,
            'val_selection_score': float(main_val_score),
            'thresholds': {
                'headline':    float(thr_headline),
                'f1_optimal':  float(thr_f1opt),
                'prior_match': float(thr_prior),
            },
            'binary_test_roc': float(bin_roc),
            'binary_test_pr':  float(bin_pr),
            'binary_test_f1':  float(m_head['f1']),
            'binary_test_fpr': float(m_head['fpr']),
            'binary_test_fnr': float(m_head['fnr']),
            'binary_test_f1_at_f1opt':  float(m_f1opt['f1']),
            'binary_test_fpr_at_f1opt': float(m_f1opt['fpr']),
            'binary_test_fnr_at_f1opt': float(m_f1opt['fnr']),
            'pred_attack_rate':     float(m_head['pred_attack_rate']),
            'true_attack_rate':     float(yb_test_i.mean()),
            'multiclass_test_macro_f1':              float(multi_mf1_raw),
            'multiclass_test_macro_f1_visible':      float(multi_mf1_visible),
            'multiclass_test_macro_f1_merged':       float(multi_mf1_merged),
            'multiclass_test_weighted_f1':           float(multi_wf1),
        },
        'ensemble': {
            'seeds': list(ensemble_seeds), 'per_seed': seed_results,
            'thresholds': {
                'headline':    float(thr_ens),
                'f1_optimal':  float(thr_f1opt_ens),
                'prior_match': float(thr_prior_ens),
            },
            'binary_test_roc': float(ens_roc),
            'binary_test_pr':  float(ens_pr),
            'binary_test_f1':  float(m_ens['f1']),
            'binary_test_fpr': float(m_ens['fpr']),
            'binary_test_fnr': float(m_ens['fnr']),
            'binary_test_roc_ci95': [float(lo), float(hi)],
            'multiclass_test_macro_f1':              float(ens_multi_mf1_raw),
            'multiclass_test_macro_f1_merged':       float(ens_multi_mf1_merged),
            'multiclass_test_weighted_f1':           float(ens_multi_wf1),
        },
        'attention': {
            'attn_dir': ATTN_DIR,
            'mean_attention': mean_attn.tolist(),
            'per_class_attention': {class_names[i]: per_class_attn[i].tolist()
                                    for i in range(NUM_CLASSES)},
            'class_counts': class_counts.tolist(),
        },
        'per_class_seed42':   per_class_rows,
        'per_class_ensemble': per_class_rows_ens,
        'fewshot': {str(k): v for k, v in fewshot_results.items()},
        'class_names': class_names,
        'split_sizes': {'train': len(train_graphs), 'val': len(val_graphs),
                        'test': len(test_graphs)},
        'sequence_counts': {'train': len(train_starts), 'val': len(val_starts),
                            'test': len(test_starts)},
        'config': {
            'sequence_length': SEQUENCE_LENGTH, 'sequence_stride': SEQUENCE_STRIDE,
            'tft_hidden_dim': TFT_HIDDEN_DIM,
            'batch_size': BATCH_SIZE, 'grad_accum': GRAD_ACCUM_STEPS,
            'num_workers': NUM_WORKERS,
            'lr': LEARNING_RATE, 'encoder_lr_mult': ENCODER_LR_MULT,
            'epochs': EPOCHS, 'patience': PATIENCE,
            'warmup_epochs': WARMUP_EPOCHS,
            'fewshot_epochs': FEWSHOT_EPOCHS,
            'fewshot_warmup': FEWSHOT_WARMUP,
            'fewshot_fractions': FRACTIONS,
            'small_label_threshold': SMALL_LABEL_THRESHOLD,
            'use_balanced_sampler': bool(use_balanced_sampler),
            'selection_criterion': 'val_macro_f1_merged',
            'threshold_protocol': 'prior_matched' if USE_PRIOR_MATCHED_THRESHOLD else 'f1_optimal',
            'fine_tune_encoder': bool(FINE_TUNE_ENCODER),
            'n_ensemble': len(ensemble_seeds),
            'max_nodes': MAX_NODES, 'max_edges': MAX_EDGES,
            'partial_ckpt_dir': PARTIAL_DIR,
            'embedding_cache': cache_train is not None,
        },
    }
    with open(REPORT_FILE, 'w') as f:
        json.dump(report, f, indent=2)
    print(f"\nCheckpoint : {CKPT_FILE}")
    print(f"Report     : {REPORT_FILE}")
    print(f"Attention  : {ATTN_DIR}")
    print(f"Partials   : {PARTIAL_DIR}")

    del main_enc, main_tft, train_loader, val_loader, test_loader
    del cache_train, cache_val, cache_test
    del str_train, str_val, str_test, train_graphs, val_graphs, test_graphs
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return report

Cell 9 — Master runner: BYOL first, then MoCo

In [10]:
def _fmt(x, nd=4):
    return f"{x:.{nd}f}" if isinstance(x, (int, float)) else "n/a"

def _g(r, *path, default=float('nan')):
    cur = r
    for p in path:
        if not isinstance(cur, dict) or p not in cur: return default
        cur = cur[p]
    return cur if isinstance(cur, (int, float)) else default


def _print_dataset_comparison(ds, r_byol, r_moco, mode_tag):
    print(f"\n  ── {ds.upper()} · {mode_tag.upper()} comparison ──")
    if r_byol.get('_skipped_no_ssl') or r_moco.get('_skipped_no_ssl') or \
       'error' in r_byol or 'error' in r_moco:
        print(f"    BYOL : {'skipped' if r_byol.get('_skipped_no_ssl') else r_byol.get('error','ok')}")
        print(f"    MoCo : {'skipped' if r_moco.get('_skipped_no_ssl') else r_moco.get('error','ok')}")
        return
    b_roc = _g(r_byol, 'main_run', 'binary_test_roc'); b_f1 = _g(r_byol, 'main_run', 'binary_test_f1')
    b_pr  = _g(r_byol, 'main_run', 'binary_test_pr')
    m_roc = _g(r_moco, 'main_run', 'binary_test_roc'); m_f1 = _g(r_moco, 'main_run', 'binary_test_f1')
    m_pr  = _g(r_moco, 'main_run', 'binary_test_pr')
    eb_roc = _g(r_byol, 'ensemble', 'binary_test_roc'); eb_f1 = _g(r_byol, 'ensemble', 'binary_test_f1')
    eb_pr  = _g(r_byol, 'ensemble', 'binary_test_pr')
    em_roc = _g(r_moco, 'ensemble', 'binary_test_roc'); em_f1 = _g(r_moco, 'ensemble', 'binary_test_f1')
    em_pr  = _g(r_moco, 'ensemble', 'binary_test_pr')
    b_mf1 = _g(r_byol, 'ensemble', 'multiclass_test_macro_f1')
    m_mf1 = _g(r_moco, 'ensemble', 'multiclass_test_macro_f1')
    b_fpr = _g(r_byol, 'ensemble', 'binary_test_fpr'); b_fnr = _g(r_byol, 'ensemble', 'binary_test_fnr')
    m_fpr = _g(r_moco, 'ensemble', 'binary_test_fpr'); m_fnr = _g(r_moco, 'ensemble', 'binary_test_fnr')

    print(f"    {'metric':<22} {'BYOL':>10} {'MoCo':>10} {'Δ (MoCo-BYOL)':>16}")
    print(f"    {'-'*22} {'-'*10} {'-'*10} {'-'*16}")

    def row(name, b, m, nd=4):
        d = m - b if isinstance(b, float) and isinstance(m, float) else float('nan')
        print(f"    {name:<22} {_fmt(b, nd):>10} {_fmt(m, nd):>10} {_fmt(d, nd):>16}")

    print("    single seed=42")
    row("  binary ROC", b_roc, m_roc); row("  binary F1",  b_f1,  m_f1); row("  binary PR",  b_pr,  m_pr)
    print("    ensemble")
    row("  ensemble ROC", eb_roc, em_roc); row("  ensemble F1",  eb_f1,  em_f1)
    row("  ensemble PR",  eb_pr,  em_pr);  row("  ensemble mF1", b_mf1,  m_mf1)
    row("  FPR",          b_fpr,  m_fpr);  row("  FNR",          b_fnr,  m_fnr)

    if isinstance(em_roc, float) and isinstance(eb_roc, float):
        winner = "MoCo" if em_roc > eb_roc else ("BYOL" if eb_roc > em_roc else "tie")
        print(f"     >  ensemble ROC winner: {winner}  (margin {abs(em_roc-eb_roc):.4f})")

def run_one_dataset(ds):
    print("\n\n" + "#" * 88)
    print(f"#  DATASET = {ds.upper()}")
    print("#" * 88)

    ALL_RESULTS.setdefault(ds, {})
    for ssl in SSL_METHODS_TO_RUN:
        print(f"\n\n{'='*88}")
        print(f"===  {ds}  ·  SSL = {ssl.upper()}  ===")
        print(f"{'='*88}")
        try:
            ALL_RESULTS[ds][ssl] = run_dataset(ds, ssl)
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f"!!! FAILED on {ds}/{ssl}: {e}")
            ALL_RESULTS[ds][ssl] = {'error': str(e),
                                    'dataset': ds,
                                    'ssl_method': ssl}
            gc.collect()
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()

    mode_tag = "finetune" if FINE_TUNE_ENCODER else "frozen"
    _print_dataset_comparison(
        ds,
        ALL_RESULTS[ds].get('byol', {}),
        ALL_RESULTS[ds].get('moco', {}),
        mode_tag,
    )
    return ALL_RESULTS[ds]


def finalize_results(save_json=True):
    global ALL_RESULTS
    ALL_RESULTS_BY_SSL = {ssl: {} for ssl in SSL_METHODS_TO_RUN}
    for ds, per_ssl in ALL_RESULTS.items():
        for ssl, r in per_ssl.items():
            ALL_RESULTS_BY_SSL[ssl][ds] = r
    ALL_RESULTS = ALL_RESULTS_BY_SSL

    print("\n" + "=" * 70)
    print("ALL_RESULTS finalised")
    print("=" * 70)
    for ssl, per_ds in ALL_RESULTS.items():
        print(f"  {ssl.upper():<6} datasets: {list(per_ds.keys())}")

    if save_json:
        import json as _json
        with open("phase3_all_results_raw.json", "w") as f:
            _json.dump(ALL_RESULTS, f, indent=2, default=str)
        print("\nSaved phase3_all_results_raw.json")

    return ALL_RESULTS


# ---- initialise accumulator (keeps prior entries if the cell is re-run) ----
if 'ALL_RESULTS' not in globals() or not isinstance(ALL_RESULTS, dict):
    ALL_RESULTS = {}
else:
    print(f"[keep] ALL_RESULTS already has: {list(ALL_RESULTS.keys())}")

print(f"ALL_RESULTS = {ALL_RESULTS}")
print("Scaffolding ready: run_one_dataset(ds) and finalize_results() defined.")

ALL_RESULTS = {}
Scaffolding ready: run_one_dataset(ds) and finalize_results() defined.


In [11]:
_ = run_one_dataset('gotham')



########################################################################################
#  DATASET = GOTHAM
########################################################################################


===  gotham  ·  SSL = BYOL  ===
Single seed: 42   Ensemble seeds: [1337, 2024, 7] (3 members)
Balanced sampler: True
Resume from partials: True  force_retrain: False
[SKIP] gotham [BYOL/finetune]: cached


===  gotham  ·  SSL = MOCO  ===
Single seed: 42   Ensemble seeds: [1337, 2024, 7] (3 members)
Balanced sampler: True
Resume from partials: True  force_retrain: False
[SKIP] gotham [MOCO/finetune]: cached

  ── GOTHAM · FINETUNE comparison ──
    metric                       BYOL       MoCo    Δ (MoCo-BYOL)
    ---------------------- ---------- ---------- ----------------
    single seed=42
      binary ROC               0.9910     0.9899          -0.0010
      binary F1                0.9697     0.9771           0.0074
      binary PR                0.9658     0.9567          -0.0091
 

In [12]:
_ = run_one_dataset('nf_bot')

gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()



########################################################################################
#  DATASET = NF_BOT
########################################################################################


===  nf_bot  ·  SSL = BYOL  ===
Single seed: 42   Ensemble seeds: [1337, 2024, 7] (3 members)
Balanced sampler: True
Resume from partials: True  force_retrain: False
[SKIP] nf_bot [BYOL/finetune]: cached


===  nf_bot  ·  SSL = MOCO  ===
Single seed: 42   Ensemble seeds: [1337, 2024, 7] (3 members)
Balanced sampler: True
Resume from partials: True  force_retrain: False
[SKIP] nf_bot [MOCO/finetune]: cached

  ── NF_BOT · FINETUNE comparison ──
    metric                       BYOL       MoCo    Δ (MoCo-BYOL)
    ---------------------- ---------- ---------- ----------------
    single seed=42
      binary ROC               0.9553     0.9737           0.0184
      binary F1                0.9728     0.9698          -0.0030
      binary PR                0.9941     0.9971           0.0030
 

In [13]:
_ = run_one_dataset('nf_unsw')

gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()



########################################################################################
#  DATASET = NF_UNSW
########################################################################################


===  nf_unsw  ·  SSL = BYOL  ===
Single seed: 42   Ensemble seeds: [1337, 2024, 7] (3 members)
Balanced sampler: True
Resume from partials: True  force_retrain: False
[SKIP] nf_unsw [BYOL/finetune]: cached


===  nf_unsw  ·  SSL = MOCO  ===
Single seed: 42   Ensemble seeds: [1337, 2024, 7] (3 members)
Balanced sampler: True
Resume from partials: True  force_retrain: False
[SKIP] nf_unsw [MOCO/finetune]: cached

  ── NF_UNSW · FINETUNE comparison ──
    metric                       BYOL       MoCo    Δ (MoCo-BYOL)
    ---------------------- ---------- ---------- ----------------
    single seed=42
      binary ROC               0.8714     0.8616          -0.0097
      binary F1                0.7806     0.8249           0.0443
      binary PR                0.8676     0.8568          -0.

In [14]:
torch.cuda.empty_cache()

In [15]:
_ = run_one_dataset('nf_ton')

gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()



########################################################################################
#  DATASET = NF_TON
########################################################################################


===  nf_ton  ·  SSL = BYOL  ===
Single seed: 42   Ensemble seeds: [1337, 2024, 7] (3 members)
Balanced sampler: True
Resume from partials: True  force_retrain: False
[SKIP] nf_ton [BYOL/finetune]: cached


===  nf_ton  ·  SSL = MOCO  ===
Single seed: 42   Ensemble seeds: [1337, 2024, 7] (3 members)
Balanced sampler: True
Resume from partials: True  force_retrain: False
[SKIP] nf_ton [MOCO/finetune]: cached

  ── NF_TON · FINETUNE comparison ──
    metric                       BYOL       MoCo    Δ (MoCo-BYOL)
    ---------------------- ---------- ---------- ----------------
    single seed=42
      binary ROC               0.9108     0.8799          -0.0308
      binary F1                0.9500     0.9380          -0.0121
      binary PR                0.9688     0.9614          -0.0074
 

In [16]:
_ = finalize_results(save_json=True)


ALL_RESULTS finalised
  BYOL   datasets: ['gotham', 'nf_bot', 'nf_unsw', 'nf_ton']
  MOCO   datasets: ['gotham', 'nf_bot', 'nf_unsw', 'nf_ton']

Saved phase3_all_results_raw.json


Cell 10 — Side-by-side comparison tables

In [17]:
def _fmt(x, nd=4):
    return f"{x:.{nd}f}" if isinstance(x, (int, float)) else "n/a"

def _g(r, *path, default=float('nan')):
    cur = r
    for p in path:
        if not isinstance(cur, dict) or p not in cur: return default
        cur = cur[p]
    return cur if isinstance(cur, (int, float)) else default

# ============ TABLE 1: single-seed ============
print("\n" + "=" * 110)
print(f"TABLE 1 · SINGLE-SEED TEST METRICS (seed=42, mode={'finetune' if FINE_TUNE_ENCODER else 'frozen'})")
print("=" * 110)
print(f"{'dataset':<10} | "
      f"{'BYOL ROC':>9} {'BYOL F1':>8} {'BYOL PR':>8} | "
      f"{'MoCo ROC':>9} {'MoCo F1':>8} {'MoCo PR':>8} | "
      f"{'Δ ROC':>8} {'Δ F1':>8}")
print("-" * 110)
for ds in DATASETS_TO_RUN:
    rb = ALL_RESULTS.get('byol', {}).get(ds, {})
    rm = ALL_RESULTS.get('moco', {}).get(ds, {})
    br = _g(rb, 'main_run', 'binary_test_roc'); bf = _g(rb, 'main_run', 'binary_test_f1')
    bp = _g(rb, 'main_run', 'binary_test_pr')
    mr = _g(rm, 'main_run', 'binary_test_roc'); mf = _g(rm, 'main_run', 'binary_test_f1')
    mp = _g(rm, 'main_run', 'binary_test_pr')
    print(f"{ds:<10} | {_fmt(br):>9} {_fmt(bf):>8} {_fmt(bp):>8} | "
          f"{_fmt(mr):>9} {_fmt(mf):>8} {_fmt(mp):>8} | "
          f"{_fmt(mr-br if isinstance(mr,float) and isinstance(br,float) else float('nan')):>8} "
          f"{_fmt(mf-bf if isinstance(mf,float) and isinstance(bf,float) else float('nan')):>8}")

# ============ TABLE 2: ensemble ============
print("\n" + "=" * 110)
print("TABLE 2 · ENSEMBLE TEST METRICS")
print("=" * 110)
print(f"{'dataset':<10} | "
      f"{'BYOL ROC':>9} {'BYOL F1':>8} {'BYOL PR':>8} | "
      f"{'MoCo ROC':>9} {'MoCo F1':>8} {'MoCo PR':>8} | "
      f"{'Δ ROC':>8} {'Δ F1':>8}")
print("-" * 110)
for ds in DATASETS_TO_RUN:
    rb = ALL_RESULTS.get('byol', {}).get(ds, {})
    rm = ALL_RESULTS.get('moco', {}).get(ds, {})
    br = _g(rb, 'ensemble', 'binary_test_roc'); bf = _g(rb, 'ensemble', 'binary_test_f1')
    bp = _g(rb, 'ensemble', 'binary_test_pr')
    mr = _g(rm, 'ensemble', 'binary_test_roc'); mf = _g(rm, 'ensemble', 'binary_test_f1')
    mp = _g(rm, 'ensemble', 'binary_test_pr')
    print(f"{ds:<10} | {_fmt(br):>9} {_fmt(bf):>8} {_fmt(bp):>8} | "
          f"{_fmt(mr):>9} {_fmt(mf):>8} {_fmt(mp):>8} | "
          f"{_fmt(mr-br if isinstance(mr,float) and isinstance(br,float) else float('nan')):>8} "
          f"{_fmt(mf-bf if isinstance(mf,float) and isinstance(bf,float) else float('nan')):>8}")

# ============ TABLE 3: error rates ============
print("\n" + "=" * 110)
print("TABLE 3 · ERROR RATES (ensemble) — lower is better")
print("=" * 110)
print(f"{'dataset':<10} | "
      f"{'BYOL FPR':>9} {'BYOL FNR':>9} | "
      f"{'MoCo FPR':>9} {'MoCo FNR':>9}")
print("-" * 110)
for ds in DATASETS_TO_RUN:
    rb = ALL_RESULTS.get('byol', {}).get(ds, {})
    rm = ALL_RESULTS.get('moco', {}).get(ds, {})
    print(f"{ds:<10} | "
          f"{_fmt(_g(rb, 'ensemble','binary_test_fpr')):>9} "
          f"{_fmt(_g(rb, 'ensemble','binary_test_fnr')):>9} | "
          f"{_fmt(_g(rm, 'ensemble','binary_test_fpr')):>9} "
          f"{_fmt(_g(rm, 'ensemble','binary_test_fnr')):>9}")

# ============ TABLE 4: multiclass ============
print("\n" + "=" * 110)
print("TABLE 4 · MULTICLASS MACRO-F1 (ensemble)")
print("=" * 110)
print(f"{'dataset':<10} | {'BYOL mF1':>10} | {'MoCo mF1':>10} | {'Δ':>8}")
print("-" * 110)
for ds in DATASETS_TO_RUN:
    rb = ALL_RESULTS.get('byol', {}).get(ds, {})
    rm = ALL_RESULTS.get('moco', {}).get(ds, {})
    bm = _g(rb, 'ensemble', 'multiclass_test_macro_f1')
    mm = _g(rm, 'ensemble', 'multiclass_test_macro_f1')
    print(f"{ds:<10} | {_fmt(bm):>10} | {_fmt(mm):>10} | "
          f"{_fmt(mm-bm if isinstance(mm,float) and isinstance(bm,float) else float('nan')):>8}")


TABLE 1 · SINGLE-SEED TEST METRICS (seed=42, mode=finetune)
dataset    |  BYOL ROC  BYOL F1  BYOL PR |  MoCo ROC  MoCo F1  MoCo PR |    Δ ROC     Δ F1
--------------------------------------------------------------------------------------------------------------
gotham     |    0.9910   0.9697   0.9658 |    0.9899   0.9771   0.9567 |  -0.0010   0.0074
nf_bot     |    0.9553   0.9728   0.9941 |    0.9737   0.9698   0.9971 |   0.0184  -0.0030
nf_unsw    |    0.8714   0.7806   0.8676 |    0.8616   0.8249   0.8568 |  -0.0097   0.0443
nf_ton     |    0.9108   0.9500   0.9688 |    0.8799   0.9380   0.9614 |  -0.0308  -0.0121

TABLE 2 · ENSEMBLE TEST METRICS
dataset    |  BYOL ROC  BYOL F1  BYOL PR |  MoCo ROC  MoCo F1  MoCo PR |    Δ ROC     Δ F1
--------------------------------------------------------------------------------------------------------------
gotham     |    0.9892   0.9697   0.9641 |    0.9899   0.9697   0.9592 |   0.0007   0.0000
nf_bot     |    0.9787   0.9722   0.9977 |    0

Cell 11 — Per-class comparison (one-vs-Benign ROC, ensemble)

In [19]:
print("\n" + "=" * 110)
print("TABLE 5 · PER-CLASS ONE-VS-BENIGN ROC (ensemble)")
print("=" * 110)
for ds in DATASETS_TO_RUN:
    rb = ALL_RESULTS.get('byol', {}).get(ds, {})
    rm = ALL_RESULTS.get('moco', {}).get(ds, {})
    pc_b = {r['class']: r for r in rb.get('per_class_ensemble', [])}
    pc_m = {r['class']: r for r in rm.get('per_class_ensemble', [])}
    classes = sorted(set(pc_b) | set(pc_m))
    if not classes: continue
    print(f"\n  {ds}")
    print(f"  {'class':<28} {'n':>6} | {'BYOL ROC':>9} {'BYOL rec':>9} | "
          f"{'MoCo ROC':>9} {'MoCo rec':>9} | {'Δ ROC':>8}")
    for c in classes:
        b = pc_b.get(c, {}); m = pc_m.get(c, {})
        br = b.get('roc_vs_benign', float('nan')); mr = m.get('roc_vs_benign', float('nan'))
        brev = b.get('recall_at_thr', float('nan')); mrev = m.get('recall_at_thr', float('nan'))
        n = b.get('n_test', m.get('n_test', 0))
        print(f"  {c[:28]:<28} {n:>6} | {_fmt(br):>9} {_fmt(brev,3):>9} | "
              f"{_fmt(mr):>9} {_fmt(mrev,3):>9} | "
              f"{_fmt(mr-br if isinstance(mr,float) and isinstance(br,float) else float('nan')):>8}")


TABLE 5 · PER-CLASS ONE-VS-BENIGN ROC (ensemble)

  gotham
  class                             n |  BYOL ROC  BYOL rec |  MoCo ROC  MoCo rec |    Δ ROC
  Mirai C&C Communication          63 |    0.9895     0.984 |    0.9902     0.984 |   0.0007
  Mirai GRE Flooding                1 |    0.9850     1.000 |    0.9850     1.000 |   0.0000
  Mirai UDP Flooding                1 |    0.9774     1.000 |    0.9774     1.000 |   0.0000

  nf_bot
  class                             n |  BYOL ROC  BYOL rec |  MoCo ROC  MoCo rec |    Δ ROC
  DDoS                            117 |    0.9995     1.000 |    0.9998     1.000 |   0.0002
  DoS                             128 |    0.9993     1.000 |    0.9998     1.000 |   0.0004
  Reconnaissance                   27 |    0.8048     0.593 |    0.8259     0.630 |   0.0212
  Theft                            40 |    0.9693     0.900 |    0.9836     0.950 |   0.0143

  nf_unsw
  class                             n |  BYOL ROC  BYOL rec |  MoCo ROC  MoCo rec 

Cell 12 — Save combined JSON

In [20]:
combined = {
    'mode': 'finetune' if FINE_TUNE_ENCODER else 'frozen',
    'datasets': DATASETS_TO_RUN,
    'ssl_methods': SSL_METHODS_TO_RUN,
    'byol': ALL_RESULTS.get('byol', {}),
    'moco': ALL_RESULTS.get('moco', {}),
}
out = f"phase3_compare_byol_vs_moco_{'finetune' if FINE_TUNE_ENCODER else 'frozen'}.json"
with open(out, 'w') as f:
    json.dump(combined, f, indent=2, default=str)
print(f"Saved combined comparison  >  {out}")

Saved combined comparison  >  phase3_compare_byol_vs_moco_finetune.json
